## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:胡煜杭


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
##本题在解决过程中求助了身边的同学，并且使用了大模型来辅助完成这道题目
import sys

# 设置递归深度限制。
# 因为后面有递归函数 build_plan，如果递归层数稍深，默认递归深度可能不够。
# 这里把递归深度调大，避免程序因为递归层数问题报错。
sys.setrecursionlimit(1000000)


class OperationRecorder:
    """
    OperationRecorder 的作用：
    这个类专门用来“模拟魔法操作”和“记录魔法操作”。

    题目中有三种魔法：

    0：交换魔法
       交换石板上所有数字 a 和 b。
       注意：a 和 b 是题目给定的两个固定数字，不能自己随便选。

    1 x：异或魔法
       把每块石板上的数字都异或 x。

    2 x：加法魔法
       把每块石板上的数字都加 x，结果对 n 取模。

    本类中：
    self.board 保存当前石板排列状态；self.ops 保存已经使用过的操作，最后需要输出。
    """

    def __init__(self, n, a, b, board):
        """
        初始化函数。

        参数：
        n：石板数量，也是数字范围 0 到 n-1。
        a：交换魔法固定数字 a。
        b：交换魔法固定数字 b。
        board：当前石板上的数字排列。
        """

        # n 表示石板总数。
        self.n = n

        # a 和 b 是交换魔法中固定可以交换的两个数字。
        self.a = a
        self.b = b

        # board[:] 表示复制一份 board，避免直接修改外部传入的原数组。
        self.board = board[:]

        # ops 用来记录所有已经使用的操作。
        # 最后输出答案时，需要输出这个列表中的所有操作。
        self.ops = []

    def add_all(self, x):
        """
        执行加法魔法：操作 2 x。

        含义：
        把每块石板上的数字都加上 x，然后对 n 取模。

        例如：
        n = 8，当前数字为 7，x = 2。那么：(7 + 2) % 8 = 1
        """

        # 因为题目规定加法结果要对 n 取模，所以 x 本身也可以先对 n 取模，避免 x 太大。
        x %= self.n

        # 如果 x = 0，说明加 0 不会改变任何东西。这种无意义操作不需要记录。
        if x == 0:
            return

        # 记录操作。2 表示加法魔法，x 表示加的数字。
        self.ops.append((2, x))

        # 真正更新当前石板状态。对 board 中每个 value，都变成 (value + x) % n。
        self.board[:] = [(value + x) % self.n for value in self.board]

    def xor_all(self, x):
        """
        执行异或魔法：操作 1 x。

        含义：
        把每块石板上的数字都异或 x。

        异或可以理解为二进制上的“不同为 1，相同为 0”。

        例如：
        5 的二进制是 101。3 的二进制是 011。5 ^ 3 = 110，也就是 6。
        """

        # 因为数字范围是 0 到 n-1，n 是 2 的幂，所以 x 对 n 取模后仍然在合法范围内。
        x %= self.n

        # 如果 x = 0，那么 value ^ 0 = value，不会改变任何数字。所以不记录无效操作。
        if x == 0:
            return

        # 记录操作。1 表示异或魔法，x 表示异或的数字。
        self.ops.append((1, x))

        # 更新当前石板状态。
        self.board[:] = [value ^ x for value in self.board]

    def swap_fixed_pair(self):
        """
        执行交换魔法：操作 0。

        题目中的交换魔法只能交换固定的两个数字 a 和 b。

        注意：
        交换的是“数字标签”，不是交换数组下标。

        例如：
        a = 0, b = 1。board = [2, 1, 3, 0]
        执行交换后：
        1 变成 0，0 变成 1。board = [2, 0, 3, 1]
        """

        # 记录交换魔法。
        self.ops.append((0,))

        # new_board 用来保存交换后的新状态。
        new_board = []

        # 遍历当前每个石板上的数字。
        for value in self.board:

            # 如果当前数字是 a，就换成 b。
            if value == self.a:
                new_board.append(self.b)

            # 如果当前数字是 b，就换成 a。
            elif value == self.b:
                new_board.append(self.a)

            # 其他数字不受交换魔法影响。
            else:
                new_board.append(value)

        # 用新状态替换原来的 board。
        self.board[:] = new_board

    def run_script(self, script):
        """
        执行一串已经设计好的操作脚本。script 是一个操作列表。

        每个操作用元组表示：
        (0, 0) 或 (0,) 表示交换魔法；(1, x) 表示异或魔法；(2, x) 表示加法魔法。
        """

        # 依次执行脚本中的每一个操作。
        for kind, value in script:

            # kind = 0，执行固定交换 a 和 b。
            if kind == 0:
                self.swap_fixed_pair()

            # kind = 1，执行异或 x。
            elif kind == 1:
                self.xor_all(value)

            # kind = 2，执行加 x。
            else:
                self.add_all(value)


def lowbit(x):
    """
    返回 x 的二进制中最低位的 1 所代表的数值。

    例如：
    x = 12。二进制为 1100。最低位的 1 在第 3 位，对应数值 4。所以 lowbit(12) = 4。

    在代码里：x & -x 是一个经典写法，用来快速得到 lowbit。
    """

    return x & -x


class ResiduePlanner:
    """
    ResiduePlanner 的作用：负责进行第一步：余数组校正。

    这里的“余数组”可以理解为按照下标对 block 取余分组。

    目标是让每个位置 i 上的数字 value 满足：value % block == i % block

    也就是说：
    位置 0, block, 2block, ... 上应该放余数为 0 的数字；
    位置 1, block+1, 2block+1, ... 上应该放余数为 1 的数字；
    依此类推。

    这样做的意义：先把数字分到正确的“余数组”里。后面再在每个余数组内部排序，会更容易。
    """

    def __init__(self, block):
        """
        block 表示分组大小。

        如果 block = 4，
        那么数字会按照对 4 取余分成 4 组：
        余数为 0 的一组；
        余数为 1 的一组；
        余数为 2 的一组；
        余数为 3 的一组。
        """

        self.block = block

    def compress_xor_ops(self, plan):
        """
        合并连续的异或操作。

        为什么可以合并？
        因为连续异或满足：

            xor x 后再 xor y
            等价于 xor (x ^ y)

        在 plan 中：正数 x 表示 add x；负数 -x 表示 xor x。

        例如：
        -3 表示 xor 3；-5 表示 xor 5；连续出现 -3, -5，可以合并成 -(3 ^ 5)。
        """

        # result 保存合并后的操作。
        result = []

        # 遍历原始操作计划。
        for op in plan:

            # 如果 result 不是空，并且最后一个操作是异或操作，当前 op 也是异或操作，就可以尝试合并。
            if result and result[-1] < 0 and op < 0:

                # result[-1] 是负数，所以 -result[-1] 才是真正的异或值。op 也是负数，所以 -op 才是真正的异或值。
                merged = (-result[-1]) ^ (-op)

                # 如果合并结果为 0，说明这两个异或操作抵消了，可以直接删除最后一个操作。
                if merged == 0:
                    result.pop()

                # 否则把最后一个异或操作更新为合并后的异或操作。
                else:
                    result[-1] = -merged

            # 如果不能合并，就直接加入结果列表。
            else:
                result.append(op)

        return result

    def build_plan(self, pattern):
        """
        根据当前低位余数 pattern 构造校正方案。

        pattern[i] 表示：第 i 个位置当前数字的低位余数。

        返回值 plan 中：正数 x 表示执行 add x；负数 -x 表示执行 xor x。

        这个函数使用递归思想：把当前问题按照奇偶位置拆成两个更小的问题。

        通俗理解：当前要调整的是若干位二进制低位。每次递归处理一半规模，相当于一层一层处理二进制位。
        """

        # size 表示当前 pattern 的长度。
        size = len(pattern)

        # 如果 size = 1，说明已经分到不能再分了，不需要操作。
        if size == 1:
            return []

        # half 是当前规模的一半。
        half = size // 2

        # even_part 处理偶数下标位置：pattern[0], pattern[2], pattern[4], ...每个值除以 2，相当于去掉最低位。
        even_part = []

        for i in range(half):
            even_part.append(pattern[2 * i] // 2)

        # odd_part 处理奇数下标位置：pattern[1], pattern[3], pattern[5], ...每个值除以 2，也相当于去掉最低位。
        odd_part = []

        for i in range(half):
            odd_part.append(pattern[2 * i + 1] // 2)

        # 递归处理偶数位置子问题。
        even_plan = self.build_plan(even_part)

        # 递归处理奇数位置子问题。
        odd_plan = self.build_plan(odd_part)

        # 如果任意一个子问题无法构造方案，则当前问题也无法完成。
        if even_plan is None or odd_plan is None:
            return None

        # plan 保存当前层构造出来的操作。
        plan = []

        # 如果 pattern[0] 的最低位是 1，说明当前最低位状态需要调整。
        if pattern[0] & 1:

            # 当 size = 2 时，用 add 1 就可以修正。
            if size == 2:
                plan.append(1)

            # 否则使用 xor 1 进行修正。
            else:
                plan.append(-1)

        # 下面处理偶数位置子问题的操作提升。
        even_mark = 0

        for op in even_plan:

            # 如果子问题中是 add 操作，提升到当前层时，需要用 xor 1 和 add 1 的组合来表达。
            if op > 0:
                plan.append(-1)
                plan.append(1)

            # 如果子问题中是 xor 操作，提升到当前层时，异或值扩大 2 倍。
            else:
                lifted = op * 2
                plan.append(lifted)

                # even_mark 用来记录该操作对高位的影响。
                even_mark ^= -lifted

        # 如果 even_mark 非 0，需要补一个异或操作抵消影响。
        if even_mark:
            plan.append(-even_mark)

        # 下面处理奇数位置子问题的操作提升。
        odd_mark = 0

        for op in odd_plan:

            # 如果子问题中是 add 操作，对奇数部分提升时，组合顺序和偶数部分相反。
            if op > 0:
                plan.append(1)
                plan.append(-1)

            # 如果子问题中是 xor 操作，同样提升为 2 倍。
            else:
                lifted = op * 2
                plan.append(lifted)

                # 记录对高位的影响。
                odd_mark ^= -lifted

        # 检查两个子问题在当前最高位上的影响是否一致。如果不一致，说明无法通过全局操作统一校正。
        if (odd_mark & half) != (even_mark & half):
            return None

        # 去掉当前最高位，只保留低位影响。
        if even_mark >= half:
            even_mark -= half

        if odd_mark >= half:
            odd_mark -= half

        # 如果偶数部分和奇数部分剩余影响仍不一致，则失败。
        if even_mark != odd_mark:
            return None

        # 合并连续异或操作，减少操作数量。
        return self.compress_xor_ops(plan)


class LabelExchanger:
    """
    LabelExchanger 的作用：构造“交换任意两个数字 x 和 y”的操作脚本。

    题目原本只允许直接交换固定的 a 和 b。但是借助：加法魔法；异或魔法；固定交换魔法；

    可以把想交换的 x 和 y 暂时映射到 a 和 b，执行固定交换，再映射回来。这样就能间接实现任意两个数字标签的交换。
    """

    def __init__(self, n, block, fixed_a, fixed_b):
        """
        n：数字范围大小。
        block：分组大小。
        fixed_a, fixed_b：题目中固定可以交换的两个数字。
        """

        self.n = n
        self.block = block
        self.a = fixed_a
        self.b = fixed_b

    def anchor_pair(self, x, y):
        """
        计算数字 x 和 y 在构造交换脚本时对应的中间锚点。

        这个函数比较数学化。

        可以简单理解为：为了用固定交换 a、b 去模拟交换 x、y，我们需要先把 x、y 通过加法和异或变换到合适的位置。anchor_pair 就是在计算这个合适的位置。
        """

        n = self.n
        block = self.block

        # delta 表示 y 相对于 x 的偏移关系。
        delta = (y - x + n - block + n) % n

        # left 和 right 是两个中间变量，用来构造锚点。
        left = 0
        right = 0

        # step 从 n/2 开始逐步减半。
        step = n // 2

        # 逐层判断 delta 落在哪一半中。
        while step >= 2 * block:

            # 如果 delta 大于当前 step，就说明目标偏移在右侧部分。
            if delta >= step:
                delta -= step
                right += step // 2

            # 否则在左侧部分。
            else:
                left += step // 2

            # step 每次减半，类似二分。
            step //= 2

        # low_part 是 x 在 block 内的低位部分。
        low_part = x & (block - 1)

        # 构造左右锚点。
        left += n // 2
        left += low_part
        right += low_part

        return left, right

    def direct_exchange_script(self, x, y):
        """
        直接构造交换 x 和 y 的脚本。

        适用条件：x 和 y 所在的大组奇偶不同。

        基本思想：
        先用加法和异或把 x、y 映射到固定的 a、b；执行一次交换魔法 0；再把映射恢复回来。
        """

        # 计算固定 a、b 对应的锚点。
        base_x, _ = self.anchor_pair(self.a, self.b)

        # 计算目标 x、y 对应的锚点。
        target_x, _ = self.anchor_pair(x, y)

        # 脚本含义：
        # 1. 平移数字，让 x 到达 target_x 附近；
        # 2. 异或变换，让 target_x 对齐 base_x；
        # 3. 平移到固定交换 a、b 的位置；
        # 4. 执行固定交换 0；
        # 5. 反向恢复前面的变换。
        script = [
            (2, (target_x - x) % self.n),
            (1, target_x ^ base_x),
            (2, (self.a - base_x) % self.n),
            (0, 0),
            (2, (base_x - self.a) % self.n),
            (1, target_x ^ base_x),
            (2, (x - target_x) % self.n),
        ]

        return script

    def exchange_script(self, x, y):
        """
        返回交换数字 x 和数字 y 所需的操作脚本。

        如果 x 和 y 所在的大组奇偶不同：可以直接构造交换脚本。

        如果 x 和 y 所在的大组奇偶相同：直接交换比较困难，需要借助一个 helper 作为中转。

        使用三次交换：
        swap(x, helper)
        swap(y, helper)
        swap(x, helper)

        这样最终效果等价于交换 x 和 y。
        """

        block = self.block

        # 计算 x 所在的大组奇偶。
        group_x = (x // block) & 1

        # 计算 y 所在的大组奇偶。
        group_y = (y // block) & 1

        # 如果大组奇偶不同，可以直接交换。
        if group_x != group_y:
            return self.direct_exchange_script(x, y)

        # 如果大组奇偶相同，需要找一个 helper。helper 要和 x 不在同一个大组奇偶中。
        if group_x == 0:
            helper = (x & (block - 1)) + block
        else:
            helper = x & (block - 1)

        # 用三次交换实现间接交换。
        script = []

        script.extend(self.exchange_script(x, helper))
        script.extend(self.exchange_script(y, helper))
        script.extend(self.exchange_script(x, helper))

        return script


class GroupSorter:
    """
    GroupSorter 的作用：在余数组校正完成后，对每个余数组内部进行排序。

    前面 ResiduePlanner 的目标是让：value % block == index % block

    这说明每个数字已经进入了正确的余数组。接下来只需要在每个余数组内部，把数字放到正确位置即可。
    """

    def __init__(self, state, exchanger, block):
        """
        state：当前操作记录器，里面保存 board 和 ops。exchanger：用于构造任意两个数字交换脚本。block：分组大小。
        """

        self.state = state
        self.exchanger = exchanger
        self.block = block

    def sort_group(self, residue):
        """
        整理某一个余数组。residue 表示余数。

        例如 block = 4：residue = 0 表示整理位置 0, 4, 8, ...。   residue = 1 表示整理位置 1, 5, 9, ...
        """

        n = self.state.n
        board = self.state.board
        block = self.block

        # 找出当前余数组中所有位置。
        positions = list(range(residue, n, block))

        # current_values 保存这些位置上当前放着的数字。
        current_values = []

        for pos in positions:
            current_values.append(board[pos])

        # 检查当前组中的数字集合是否正确。如果这一组位置应该放 {residue, residue+block, ...}，但当前数字不是这些，说明前面的余数组校正失败。
        if sorted(current_values) != positions:
            return False

        # 建立“数字 -> 当前所在位置”的映射。这样可以快速知道某个正确数字现在在哪里。
        value_position = {}

        for pos in positions:
            value_position[board[pos]] = pos

        # 依次把每个位置放成正确数字。
        for pos in positions:

            # 如果当前位置已经放对了，就不用管。
            if board[pos] == pos:
                continue

            # correct_value 是当前位置应该放的正确数字。
            correct_value = pos

            # wrong_value 是当前位置现在错误放着的数字。
            wrong_value = board[pos]

            # 找到 correct_value 现在在哪个位置。
            correct_value_position = value_position[correct_value]

            # 构造交换 correct_value 和 wrong_value 的操作脚本。
            script = self.exchanger.exchange_script(correct_value, wrong_value)

            # 执行这串操作。
            self.state.run_script(script)

            # 执行交换后，维护数字位置表。
            # correct_value 现在被换到了 pos。
            value_position[correct_value] = pos

            # wrong_value 被换到了原来 correct_value 所在的位置。
            value_position[wrong_value] = correct_value_position

        # 这一组整理成功。
        return True


def solve():
    """
    主函数。

    整体算法流程：

    第一步：读入数据。

    第二步：根据固定交换数字 a 和 b 的差值，计算 block。block 决定了我们能把数字按什么粒度进行分组。

    第三步：余数组校正。目标是让每个位置 i 上的数字满足：board[i] % block == i % block

    第四步：每个余数组内部排序。利用 LabelExchanger 构造任意两个数字的交换，把每个位置放成正确数字。

    第五步：检查是否已经变成升序排列 [0,1,2,...,n-1]。

    第六步：检查操作次数是否超过 32768。

    第七步：输出操作方案。
    """

    # 从标准输入读取所有整数。
    data = list(map(int, sys.stdin.buffer.read().split()))

    # 如果没有输入，直接结束。
    if not data:
        return

    # 第一个数是 n。
    n = data[0]

    # 第二、三个数是固定交换魔法中的 a 和 b。
    a = data[1]
    b = data[2]

    # 后面 n 个数是当前石板排列。
    board = data[3:3 + n]

    # 创建操作记录器。
    state = OperationRecorder(n, a, b, board)

    # 计算 a 和 b 的差值。因为数字都在模 n 意义下变化，所以这里对 n 取模。
    diff = (a - b + n) % n

    # block 取 diff 的 lowbit。它表示当前可控分组的最小单位。
    block = lowbit(diff)

    # 如果 block = 0，说明 diff = 0。这种情况下设置 block = n。
    if block == 0:
        block = n

    # 第一步：余数组校正。如果 block > 1，说明需要先把低位余数调整正确。
    if block > 1:

        # pattern 保存前 block 个位置上的低位余数。
        pattern = []

        for i in range(block):
            pattern.append(state.board[i] & (block - 1))

        # 创建余数组校正器。
        planner = ResiduePlanner(block)

        # 根据 pattern 构造操作计划。
        plan = planner.build_plan(pattern)

        # 如果无法构造计划，说明无解。
        if plan is None:
            print(-1)
            return

        # 执行操作计划。
        for op in plan:

            # 正数表示 add。
            if op > 0:
                state.add_all(op)

            # 负数表示 xor。
            else:
                state.xor_all(-op)

    # 第二步：分组内部排序。

    # exchanger 用来构造任意两个数字的交换脚本。
    exchanger = LabelExchanger(n, block, a, b)

    # sorter 用来整理每个余数组内部。
    sorter = GroupSorter(state, exchanger, block)

    # 对每一个余数组进行排序。
    for residue in range(block):

        # 如果某一组无法排序，则无解。
        if not sorter.sort_group(residue):
            print(-1)
            return

    # 第三步：最终检查。
    # 如果最终 board 不是 [0,1,2,...,n-1]，说明失败。
    if state.board != list(range(n)):
        print(-1)
        return

    # 题目要求操作次数不能超过 32768。
    if len(state.ops) > 32768:
        print(-1)
        return

    # 第四步：输出答案。

    # 第一行输出操作总数。
    output = [str(len(state.ops))]

    # 逐条输出操作。
    for op in state.ops:

        # 交换魔法输出 0。
        if op[0] == 0:
            output.append("0")

        # 异或魔法输出 1 x。
        elif op[0] == 1:
            output.append("1 " + str(op[1]))

        # 加法魔法输出 2 x。
        else:
            output.append("2 " + str(op[1]))

    # 输出所有内容。
    sys.stdout.write("\n".join(output))


# Python 程序入口。
# 当直接运行该文件时，会执行 solve()。
if __name__ == "__main__":
    solve()

## B 长跑

In [ ]:
import sys
from collections import deque


def can_finish(N, L, Maxn, S, stations):
    """
    判断小明能否在硬币数量不超过 S 的情况下跑到终点。

    ------------------------------------------------------------
    算法核心思路：
    ------------------------------------------------------------

    小明一开始体力是满的，每次到补给点补充体力后，体力也会恢复到 Maxn。

    因此，可以把问题看成：

        从起点 0 出发；
        每次最多向前跑 Maxn 的距离；
        中途可以在补给点花费硬币补满体力；
        问最后能否在花费不超过 S 的情况下到达终点 L。

    ------------------------------------------------------------
    状态理解：
    ------------------------------------------------------------

    如果小明已经在某个位置补满体力，那么他接下来最多还能跑 Maxn 的距离。

    所以我们只需要维护一些“已经能到达，并且已经补满体力的位置”。

    对于每个补给点 pos，如果前面存在某个位置 pre 满足：pos - pre <= Maxn

    那么说明小明可以从 pre 跑到 pos。到达 pos 后，如果选择补给，就需要再花 cost 个硬币。

    ------------------------------------------------------------
    单调队列优化思路：
    ------------------------------------------------------------

    队列 q 中保存的是当前有效的状态：(位置, 到这个位置并补满体力后的最少花费)

    队列满足两个特点：

        1. 队首是当前能到达的位置中，花费最少的状态；
        2. 队列中的花费从小到大排列。

    这样处理每个补给点时，就不用枚举前面所有补给点，直接用队首计算当前补给点的最小花费即可。

    ------------------------------------------------------------
    时间复杂度：
    ------------------------------------------------------------

    补给点排序需要 O(N log N)；每个补给点最多进队一次、出队一次，所以队列处理是 O(N)。

    总时间复杂度为 O(N log N)。
    """

    # 如果初始满体力就可以直接跑到终点，不需要任何补给
    if L <= Maxn:
        return True

    # ------------------------------------------------------------
    # 第一步：处理同一个位置有多个补给点的情况
    # ------------------------------------------------------------
    # 如果同一个位置有多个补给点，那么只需要保留花费最少的那个。因为位置相同，补给效果相同，花费越少越优。
    #
    # 只保留 0 和 L 之间的补给点：
    # 1. 位置小于等于 0 的补给点没有意义；
    # 2. 位置大于等于 L 的补给点也没有必要使用，因为到达 L 就已经成功。
    # ------------------------------------------------------------
    min_cost_at_pos = {}

    for p, c in stations:
        if 0 < p < L:
            if p not in min_cost_at_pos:
                min_cost_at_pos[p] = c
            else:
                min_cost_at_pos[p] = min(min_cost_at_pos[p], c)

    # 将补给点按照位置从小到大排序
    points = sorted(min_cost_at_pos.items())

    # ------------------------------------------------------------
    # 第二步：初始化单调队列
    # ------------------------------------------------------------
    # 起点 0 可以看成一个特殊位置：小明在这里体力是满的，并且已经花费 0 个硬币。
    #
    # 队列中每个元素的格式是：(位置, 到该位置并补满体力后的最少花费)
    # ------------------------------------------------------------
    q = deque()
    q.append((0, 0))

    # ------------------------------------------------------------
    # 第三步：从左到右处理每一个补给点
    # ------------------------------------------------------------
    for pos, cost in points:

        # --------------------------------------------------------
        # 删除队首中已经无法到达当前补给点的状态。
        #
        # 如果：pos - q[0][0] > Maxn
        #
        # 说明从 q[0] 这个位置补满体力后，也跑不到当前 pos。而后面的补给点位置只会更远，所以这个状态以后也没有用了。
        # --------------------------------------------------------
        while q and pos - q[0][0] > Maxn:
            q.popleft()

        # --------------------------------------------------------
        # 如果队列为空，说明当前补给点无法从任何有效位置到达。由于补给点已经按位置排序，后面的点更远，也无法被当前已有状态到达。
        # --------------------------------------------------------
        if not q:
            break

        # --------------------------------------------------------
        # 队首 q[0] 是当前可以到达 pos 的状态中，花费最少的。因此，到达当前补给点并在这里补满体力后的最少花费为：
        #
        #     q[0][1] + cost
        # --------------------------------------------------------
        new_money = q[0][1] + cost

        # --------------------------------------------------------
        # 如果补给后总花费已经超过 S，则这个方案不能作为有效方案。因为后续再补给只会花费更多硬币，所以不加入队列。
        # --------------------------------------------------------
        if new_money > S:
            continue

        # --------------------------------------------------------
        # 维护队列的单调性。如果队尾状态的花费 >= 当前状态的花费，那么队尾状态可以被当前状态替代。
        #
        # 原因：当前补给点的位置更靠右，花费还更少或相等。对于后面的补给点来说，当前状态一定不会比队尾状态差。
        # --------------------------------------------------------
        while q and q[-1][1] >= new_money:
            q.pop()

        # 将当前补给点作为新的有效状态加入队列
        q.append((pos, new_money))

    # ------------------------------------------------------------
    # 第四步：判断是否能从某个有效状态直接跑到终点
    # ------------------------------------------------------------
    # 队列中保存的状态都是：已经可以到达，并且已经补满体力，且花费不超过 S 的位置。
    #
    # 如果某个位置 pos 满足：L - pos <= Maxn。说明小明可以从这个位置直接跑到终点。
    # ------------------------------------------------------------
    while q and L - q[0][0] > Maxn:
        q.popleft()

    # 队列中还存在有效状态，说明可以到达终点
    return len(q) > 0


def main():
    """
    主函数：负责读取输入数据并输出答案。

    题目说明中没有给出测试组数 T，因此采用读到文件结束的方式处理多组数据。

    每组数据格式为：

        N L Maxn S
        P1 C1
        P2 C2
        ...
        PN CN
    """

    # 一次性读取全部输入
    data = sys.stdin.read().strip().split()

    # 如果没有输入，直接结束程序
    if not data:
        return

    # 将输入数据全部转换为整数
    data = list(map(int, data))

    # idx 表示当前读取到 data 列表中的位置
    idx = 0

    # 保存每组数据的输出结果
    answers = []

    # 只要还有数据，就继续读取下一组
    while idx < len(data):

        # 读取每组数据的第一行：N, L, Maxn, S
        N = data[idx]
        L = data[idx + 1]
        Maxn = data[idx + 2]
        S = data[idx + 3]
        idx += 4

        # 读取 N 个补给点
        stations = []

        for _ in range(N):
            Pi = data[idx]
            Ci = data[idx + 1]
            idx += 2

            stations.append((Pi, Ci))

        # 判断这一组数据能否成功跑到终点
        if can_finish(N, L, Maxn, S, stations):
            answers.append("Yes")
        else:
            answers.append("No")

    # 每组数据输出一行
    print("\n".join(answers))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
# 本题在题目理解和代码思路方面借助了大模型。
import sys
#============================================================
# 题目理解：
# ============================================================
# 题目给定两个长度都为 n 的字符串 A 和 B。
#
# 我们可以：
#   从 A 中选择一个可以为空的子串 A[l1 ... r1]，从 B 中选择一个可以为空的子串 B[l2 ... r2]
#
# 并且要求：r1 = l2，也就是说，可以理解为选择一个公共位置 k：
#
#   A[l ... k] + B[k ... r]
#
# 然后把这两段拼接起来。
#
# 目标：求通过这种方式能够得到的最长回文串长度。
# 注意：题目要求的是最长回文串的长度，不是回文串的个数。
#
# ============================================================
# 算法总体思路
# ============================================================
# 如果直接枚举 A 的起点、终点，再枚举 B 的起点、终点，复杂度会很高。
#
# 因为 n 最大可以到 100000，所以必须使用更快的方法。
#
# 本代码主要使用三个工具：
#
#   1. Manacher 算法
#      用来快速求出一个字符串内部所有位置的最长回文半径。
#
#   2. 字符串哈希
#      用来快速判断两个子串是否相等。
#
#   3. 二分查找
#      用来快速判断 A 向左、B 向右最多能匹配多长。
#
# 核心思想：
#
#   假设回文中心在 A 中。
#
#   那么拼接后的回文串可以看成：
#
#       A 左侧扩展部分 + A 内部回文部分 + B 右侧扩展部分
#
#   其中：
#       A 内部回文部分可以用 Manacher 算法求出；A 左侧扩展部分和 B 右侧扩展部分必须镜像相等；这部分用哈希 + 二分查找求最长匹配长度。
#
#   但是回文中心也可能在 B 中。
#
#   所以代码会计算两次：
#
#       calc_one_side(A, B)：表示回文中心在 A 中。
#
#       calc_one_side(B[::-1], A[::-1])：表示回文中心在 B 中。
#
#   最后答案取两者的最大值。
#
# ============================================================
# 时间复杂度分析
# ============================================================
# Manacher 算法需要 O(n)。
# 构造哈希数组需要 O(n)。
# 每个回文中心会尝试扩展一次，一共有 O(n) 个中心。
# 每次扩展使用二分查找，需要 O(log n) 次比较。
# 每次比较用哈希完成，时间是 O(1)。
#
# 所以 calc_one_side 的时间复杂度是：O(n log n)
#
# calc_one_side 会调用两次，所以总时间复杂度仍然是：O(n log n)


# ============================================================
# 64 位哈希相关参数
# ============================================================

# MASK 用来模拟 64 位无符号整数的自然溢出。Python 的整数不会自动溢出，所以这里用 & MASK 保留低 64 位。
MASK = (1 << 64) - 1

# BASE 是哈希进制。可以把字符串哈希理解成把字符串当作一个很大的进制数。BASE 取一个较大的奇数，一般可以降低哈希冲突概率。
BASE = 911382323


def manacher(s):
    """
    Manacher 算法。

    功能：
        在线性时间 O(n) 内，求出字符串 s 中每个位置作为回文中心时的最长回文半径。

    返回：
        odd:odd[i] 表示以 s[i] 为中心的最长奇数回文半径。

            如果 odd[i] = r，说明：s[i-r+1 ... i+r-1]，是一个回文串。

            这个回文串长度为：2*r - 1

        even:even[i] 表示以 s[i-1] 和 s[i] 中间为中心的最长偶数回文半径。

            如果 even[i] = r，说明：s[i-r ... i+r-1]，是一个回文串。

            这个回文串长度为：2*r
    """

    n = len(s)

    # ========================================================
    # 第一部分：计算奇数长度回文半径 odd
    # ========================================================

    # odd[i] 保存以 s[i] 为中心的最长奇数回文半径。
    odd = [0] * n

    # left 和 right 表示当前已经找到的、最靠右的回文区间 [left, right]。初始时还没有任何回文区间，所以 right = -1。
    left, right = 0, -1

    # 枚举每一个位置 i，把它作为奇数回文的中心。
    for i in range(n):

        # 如果 i 已经超过当前已知回文区间的右边界，说明不能利用之前的信息，只能从半径 1 开始。半径 1 表示至少包含自己这个字符。
        if i > right:
            radius = 1

        # 如果 i 在当前已知回文区间内部，可以利用回文的对称性减少重复比较。
        else:
            # mirror 是 i 关于当前回文区间中心的对称位置。
            mirror = left + right - i

            # odd[mirror] 是对称位置的回文半径。right - i + 1 是 i 到当前回文区间右边界的距离。两者取较小值作为初始半径。
            radius = min(odd[mirror], right - i + 1)

        # 从当前半径继续向左右扩展。只要左右两边没有越界，并且字符相同，就继续扩大半径。
        while (
            i - radius >= 0
            and i + radius < n
            and s[i - radius] == s[i + radius]
        ):
            radius += 1

        # 记录以 i 为中心的最长奇数回文半径。
        odd[i] = radius

        # 如果当前回文区间的右端点超过了 right，就更新当前最靠右的回文区间。
        if i + radius - 1 > right:
            left = i - radius + 1
            right = i + radius - 1

    # ========================================================
    # 第二部分：计算偶数长度回文半径 even
    # ========================================================

    # even[i] 表示以 s[i-1] 和 s[i] 中间为中心的最长偶数回文半径。
    even = [0] * n

    # 重新初始化当前最右回文区间。
    left, right = 0, -1

    # 枚举每一个偶数回文中心。
    # 中心在 s[i-1] 和 s[i] 中间。
    for i in range(n):

        # 如果 i 超过当前已知回文区间的右边界，说明没有可利用信息，偶数回文初始半径为 0。
        if i > right:
            radius = 0

        # 如果 i 在当前回文区间内，可以利用对称信息。
        else:
            # 偶数回文的镜像位置计算方式稍微不同。
            mirror = left + right - i + 1
            radius = min(even[mirror], right - i + 1)

        # 继续向左右扩展偶数回文。
        while (
            i - radius - 1 >= 0
            and i + radius < n
            and s[i - radius - 1] == s[i + radius]
        ):
            radius += 1

        # 记录偶数回文半径。
        even[i] = radius

        # 更新当前最靠右的偶数回文区间。
        if i + radius - 1 > right:
            left = i - radius
            right = i + radius - 1

    return odd, even


def build_hash(s):
    """
    构造字符串 s 的前缀哈希数组。h[i] 的含义：h[i] 表示 s[0 ... i-1] 这一段字符串的哈希值。

    为什么需要哈希？
        后面需要频繁判断两个子串是否相等。如果直接逐字符比较，长度为 L 的子串需要 O(L) 时间。

        使用前缀哈希后，可以在 O(1) 时间内得到任意子串哈希，从而快速判断两个子串是否相等。
    """

    n = len(s)

    # h 长度为 n + 1，h[0] 表示空串哈希。
    h = [0] * (n + 1)

    # 逐个字符构造前缀哈希。
    for i, ch in enumerate(s):

        # ord(ch) 把字符转换成整数编码。h[i + 1] = h[i] * BASE + ord(ch)
        #
        # 这类似于十进制数追加一位：原数字 * 10 + 新数字。这里只是把 10 换成了 BASE。
        h[i + 1] = (h[i] * BASE + ord(ch)) & MASK

    return h


def get_hash(h, power, left, right):
    """
    获取子串 s[left ... right-1] 的哈希值。

    注意：right 是开区间，不包含 right。

    例如：
        s = "abcdef"
        left = 1
        right = 4
        表示子串 s[1:4]，也就是 "bcd"。

    哈希计算公式：
        hash(left, right)= h[right] - h[left] * BASE^(right-left)
    """

    return (h[right] - (h[left] * power[right - left] & MASK)) & MASK


def calc_one_side(primary, secondary, power):
    """
    计算“回文中心在 primary 这一侧”的最长回文长度。

    对原题来说，可以先理解为：
        primary = A
        secondary = B

    我们要找的拼接形式可以理解为：
        A[l ... k] + B[k ... r]

    如果回文中心在 A 中，那么回文结构可以看成：

        A 左侧扩展部分 + A 内部回文部分 + B 右侧扩展部分

    其中：
        1. A 内部回文部分完全在 primary 中；
        2. A 左侧扩展部分从 primary 中向左取；
        3. B 右侧扩展部分从 secondary 中向右取；
        4. 左右扩展部分必须镜像相等。

    所以本函数做两件事：
        1. 用 Manacher 找 primary 内部的回文中心；
        2. 用哈希 + 二分判断两侧最多还能扩展多长。
    """

    n = len(primary)

    # 对 primary 求 Manacher 数组。odd 负责奇数长度回文。even 负责偶数长度回文。
    odd, even = manacher(primary)

    # 因为我们要比较：
    #
    #   primary 从某个位置向左走
    #   secondary 从某个位置向右走
    #
    # 但是哈希更适合比较正向连续子串。
    #
    # 所以把 primary 反转。这样 primary 向左走，就变成 reverse_primary 向右走。
    reverse_primary = primary[::-1]

    # 构造两个字符串的哈希数组。
    hash_reverse_primary = build_hash(reverse_primary)
    hash_secondary = build_hash(secondary)

    # answer 保存当前找到的最大回文长度。
    answer = 0

    def lcp_left_right(left_pos, right_pos):
        """
        求 primary 向左、secondary 向右的最长匹配长度。

        也就是求最大的 length，使得：

            primary[left_pos]       == secondary[right_pos]
            primary[left_pos - 1]   == secondary[right_pos + 1]
            primary[left_pos - 2]   == secondary[right_pos + 2]
            ...

        这正好对应回文串左右两边的镜像匹配。

        做法：使用二分查找匹配长度。每次用哈希 O(1) 判断两段字符串是否相等。
        """

        # 如果位置越界，无法匹配。
        if (
            left_pos < 0
            or right_pos < 0
            or left_pos >= n
            or right_pos >= n
        ):
            return 0

        # 如果第一个字符都不相等，最长匹配长度就是 0。
        if primary[left_pos] != secondary[right_pos]:
            return 0

        # primary[left_pos] 在 reverse_primary 中的位置。
        #
        # 原字符串 primary 的下标：0, 1, 2, ..., n - 1
        #
        # 反转后 reverse_primary 的对应下标：n - 1, n - 2, ..., 0
        #
        # 所以 primary[left_pos] 对应：reverse_primary[n - 1 - left_pos]
        start_in_reverse = n - 1 - left_pos

        # secondary 是正向比较，所以起点不变。
        start_in_secondary = right_pos

        # 匹配长度至少是 1，因为前面已经判断首字符相等。
        low = 1

        # 最大匹配长度不能超过：
        #   primary 左边剩余字符数 left_pos + 1
        #   secondary 右边剩余字符数 n - right_pos
        high = min(left_pos + 1, n - right_pos)

        # 二分查找最大匹配长度。
        while low < high:

            # 取偏右中点，避免死循环。
            mid = (low + high + 1) // 2

            # 取 primary 向左长度为 mid 的部分。在 reverse_primary 中，这一段变成正向连续子串。
            hash1 = get_hash(
                hash_reverse_primary,
                power,
                start_in_reverse,
                start_in_reverse + mid
            )

            # 取 secondary 向右长度为 mid 的部分。
            hash2 = get_hash(
                hash_secondary,
                power,
                start_in_secondary,
                start_in_secondary + mid
            )

            # 如果哈希相同，认为这两段相等，可以尝试更长。
            if hash1 == hash2:
                low = mid

            # 如果哈希不同，说明 mid 太长，需要缩短。
            else:
                high = mid - 1

        return low

    # ========================================================
    # 情况一：回文中心是 primary 中的某一个字符
    # 也就是奇数长度回文。
    # ========================================================

    for center in range(n):

        # radius 是以 center 为中心的最长奇数回文半径。
        radius = odd[center]

        # 奇数回文长度 = 2 * 半径 - 1。
        middle_len = 2 * radius - 1

        # 中间回文部分的右端点。
        right_end = center + radius - 1

        # 中间回文部分左边第一个可以继续扩展的位置。
        left_next = center - radius

        # 只使用 primary 内部这个回文串，也可以作为答案。
        if middle_len > answer:
            answer = middle_len

        # 尝试继续向两边扩展：左边从 primary[left_next] 往左；右边从 secondary[right_end] 往右。
        extra = lcp_left_right(left_next, right_end)

        # 每扩展一个匹配字符，左右各增加 1 个字符，所以总长度增加 2 * extra。
        total_len = middle_len + 2 * extra

        # 更新答案。
        if total_len > answer:
            answer = total_len

        # 理论最大长度是 n + 1。如果已经达到最大值，可以提前返回。
        if answer == n + 1:
            return answer

    # ========================================================
    # 情况二：回文中心在 primary 的两个字符中间
    # 也就是偶数长度回文。
    # ========================================================

    # center 表示中心在 primary[center-1] 和 primary[center] 中间。
    #
    # 这里 center 取到 n，是为了处理“primary 内部回文为空”的情况。
    for center in range(n + 1):

        # 如果 center < n，可以直接取 even[center]。
        if center < n:
            radius = even[center]

        # 如果 center = n，说明中心在最后一个字符后面，primary 内部没有回文部分，半径为 0。
        else:
            radius = 0

        # 偶数回文长度 = 2 * 半径。
        middle_len = 2 * radius

        # 中间回文部分的右端点。
        right_end = center + radius - 1

        # 中间回文部分左边第一个可以继续扩展的位置。
        left_next = center - radius - 1

        # 更新答案。
        if middle_len > answer:
            answer = middle_len

        # 尝试继续向两边扩展。
        extra = lcp_left_right(left_next, right_end)

        # 扩展后的总长度。
        total_len = middle_len + 2 * extra

        # 更新答案。
        if total_len > answer:
            answer = total_len

        # 如果达到理论最大值，提前返回。
        if answer == n + 1:
            return answer

    return answer


def solve():
    """
    主函数。

    步骤：
        1. 读入 n、A、B；
        2. 预处理 BASE 的幂；
        3. 计算回文中心在 A 中的最大长度；
        4. 计算回文中心在 B 中的最大长度；
        5. 输出两个答案中的较大值。
    """

    # 读取所有输入。
    data = sys.stdin.read().strip().split()

    # 第一个输入是 n。
    n = int(data[0])

    # 第二个输入是字符串 A。
    A = data[1]

    # 第三个输入是字符串 B。
    B = data[2]

    # power[i] 表示 BASE^i。
    # 用于 O(1) 计算子串哈希。
    power = [1] * (n + 1)

    for i in range(n):
        power[i + 1] = (power[i] * BASE) & MASK

    # 情况一：回文中心在 A 中。
    ans1 = calc_one_side(A, B, power)

    # 如果已经达到理论最大长度 n + 1，就不需要再计算第二种情况。
    if ans1 == n + 1:
        print(ans1)
        return

    # 情况二：回文中心在 B 中。
    #
    # 通过把 B 和 A 都反过来，把“中心在 B 中”的情况转化为“中心在第一个字符串中”的情况。
    ans2 = calc_one_side(B[::-1], A[::-1], power)

    # 输出两种情况的最大值。
    print(max(ans1, ans2))


if __name__ == "__main__":
    solve()

## D 优惠券

In [ ]:
import sys
#本题在题目分析和代码编写方面求助了大模型
# ============================================================
# 算法分析
# ============================================================
# 本题是一个“优惠券购买与使用日志修复”问题。
#
# 日志中有三种记录：
#
#   I x ：表示购买编号为 x 的优惠券；
#   O x ：表示使用编号为 x 的优惠券；
#   ?   ：表示这一行日志损坏，可以被解释成任意一次购买或使用操作。
#
# 业务规则要求：每一张优惠券必须先购买，再使用。
#
# 也就是说，对于同一个编号 x，合法的操作顺序应该是：I x -> O x -> I x -> O x -> ...
#
# 如果出现以下情况，就说明当前日志暂时不合法：
#
#   1. 当前没有购买过 x，却出现了 O x；  2. 当前已经购买了 x 但还没使用，又出现了 I x。
#
# 但是由于日志中存在 ?，所以当遇到不合法情况时，可以尝试把之前某个还没有使用过的 ? 补成缺失的操作。
#
# 例如：
#
#   ? 
#   O 1
#
# 可以把 ? 解释成 I 1，这样 O 1 就合法了。
#
# 又例如：
#
#   I 1
#   ?
#   I 1
#
# 可以把 ? 解释成 O 1，这样第二个 I 1 就合法了。
#
# ------------------------------------------------------------
# 一、状态设计
# ------------------------------------------------------------
# 为了判断每个优惠券编号当前是否合法，需要维护两个字典：
#
#   state[x]
#
# 表示编号为 x 的优惠券当前状态。
#
#   state[x] = 0：表示当前没有一张 x 处于“已购买但未使用”的状态。
#
#   state[x] = 1：表示当前有一张 x 已经购买，但是还没有使用。
#
# 如果某个 x 没有在 state 中出现过，就默认 state[x] = 0。
#
# ------------------------------------------------------------
#   last[x]
#
# 表示编号 x 上一次相关操作出现的位置。
#
# 这里的“相关操作”包括：1. 日志中真实出现的 I x 或 O x；2. 用 ? 补出来的 I x 或 O x。
#
# 为什么需要 last[x]？
#
# 因为如果要用某个 ? 来补编号 x 的操作，这个 ? 必须出现在 last[x] 之后。否则就会破坏时间顺序。
#
# 例如:第 3 行出现 I 1;第 5 行又出现 I 1
#
# 如果要补一个 O 1，它必须出现在第 3 行之后、第 5 行之前。不能用第 2 行的 ? 去补，因为第 2 行发生在第 3 行之前。
#
# ------------------------------------------------------------
# 二、为什么使用树状数组
# ------------------------------------------------------------
# 本题中，我们经常需要做一件事情：
#
#   在 last[x] 之后，找到一个还没有被使用过的 ?。
#
# 如果每次都从 last[x] + 1 开始往后线性查找，最坏情况下会非常慢，可能达到 O(m^2)。因此使用树状数组 Fenwick Tree 来维护所有还没有被使用的 ? 的位置。
#
# 树状数组支持以下操作：
#
#   add(i, 1)：
#       表示第 i 行出现了一个可用的 ?。
#
#   add(i, -1)：
#       表示第 i 行的 ? 已经被使用，不能再次使用。
#
#   sum(i)：
#       查询第 1 行到第 i 行之间还有多少个可用的 ?。
#
#   kth(k)：
#       找到当前第 k 个可用的 ? 的位置。
#
# 当需要寻找 last[x] 之后的第一个可用 ? 时： before = bit.sum(last[x])。表示 last[x] 之前和 last[x] 位置之前已经有多少个可用 ?。
#
# 如果当前总可用 ? 数量为：total = bit.sum(m)
#
# 那么如果：total - before <= 0。说明 last[x] 后面没有可用的 ?，当前行无法修复。
#
# 否则，last[x] 后面的第一个可用 ? 就是整体中的第 before + 1 个可用 ?。
#
# 因此可以通过：q_pos = bit.kth(before + 1)找到这个 ? 的位置，并用：bit.add(q_pos, -1)将它标记为已经使用。
#
# ------------------------------------------------------------
# 三、具体处理规则
# ------------------------------------------------------------
# 1. 遇到 ?
#
#   如果当前还没有发现错误，就把这一行的位置加入树状数组。因为现在还不知道这个 ? 要补成什么操作，所以先保存起来，等后面需要时再使用。
#
# ------------------------------------------------------------
# 2. 遇到 I x
#
#   I x 表示购买编号为 x 的优惠券。
#
#   如果 state[x] == 0：说明当前没有未使用的 x，可以直接购买。
#       执行后令：
#           state[x] = 1
#           last[x] = 当前行号
#
#   如果 state[x] == 1：说明之前已经购买了 x，但还没有使用。此时再次购买 x 不合法。
#       为了修复它，需要在 last[x] 之后找一个可用的 ?，把这个 ? 补成 O x。
#
#       如果找到了 ?：当前 I x 就可以合法执行；执行后 state[x] 仍然为 1；更新 last[x] 为当前行号。
#
#       如果找不到 ?：当前行就是最早无法修复的错误行。
#
# ------------------------------------------------------------
# 3. 遇到 O x
#
#   O x 表示使用编号为 x 的优惠券。
#
#   如果 state[x] == 1：说明之前已经购买过 x，并且还没有使用。所以当前可以直接使用。
#       执行后令：
#
#           state[x] = 0
#           last[x] = 当前行号
#
#   如果 state[x] == 0：说明当前没有可用的 x，却要使用 x。这是不合法的。
#
#       为了修复它，需要在 last[x] 之后找一个可用的 ?，把这个 ? 补成 I x。
#
#       如果找到了 ?：当前 O x 就可以合法执行；执行后 state[x] 仍然为 0；更新 last[x] 为当前行号。
#
#       如果找不到 ?：当前行就是最早无法修复的错误行。
#
# ------------------------------------------------------------
# 四、为什么这样做是正确的
# ------------------------------------------------------------
# 对于每个编号 x 来说，合法日志只需要满足购买和使用交替出现。
#
# state[x] 正好记录了当前是否存在一张“已购买但未使用”的 x。
#
# 当当前操作与 state[x] 匹配时，可以直接执行。
#
# 当当前操作与 state[x] 冲突时，说明中间缺少了一次相反操作。
#
#   如果 I x 冲突，说明缺少一次 O x；
#   如果 O x 冲突，说明缺少一次 I x。
#
# 此时只要能在 last[x] 之后找到一个还未使用的 ?，就可以把它补成这次缺少的操作，从而恢复合法顺序。
#
# 选择 last[x] 之后最早的 ? 是合理的：因为越早使用越不影响后面的记录；同时也保证了时间顺序不被破坏。
#
# 如果 last[x] 之后没有任何可用 ?，那么无论如何都无法补出缺少的操作，当前行就是最早错误行。
#
# ------------------------------------------------------------
# 五、时间复杂度分析
# ------------------------------------------------------------
# 设一组日志共有 m 行。
#
# 每一行最多进行以下操作：
#
#   1. 如果是 ?，执行一次 bit.add；
#   2. 如果是 I x 或 O x，可能执行 bit.sum、bit.kth、bit.add。
#
# 树状数组中的 add、sum、kth 操作时间复杂度都是 O(log m)。
#
# 因此，处理一行日志的时间复杂度最多是 O(log m)。
#
# 一共有 m 行日志，所以总时间复杂度为：O(m log m)

class Fenwick:
    """树状数组，用来维护还没有被使用的 ? 的位置。"""

    def __init__(self, n: int):
        self.n = n
        self.bit = [0] * (n + 2)

    def add(self, i: int, delta: int) -> None:
        """在位置 i 增加 delta。"""
        while i <= self.n:
            self.bit[i] += delta
            i += i & -i

    def sum(self, i: int) -> int:
        """查询 1 到 i 的 ? 数量。"""
        res = 0
        while i > 0:
            res += self.bit[i]
            i -= i & -i
        return res

    def kth(self, k: int) -> int:
        """
        找到最小的位置 pos，使得前缀和 >= k。
        也就是找到第 k 个还没有被使用的 ?。
        """
        pos = 0
        step = 1 << (self.n.bit_length() - 1)

        while step:
            nxt = pos + step
            if nxt <= self.n and self.bit[nxt] < k:
                pos = nxt
                k -= self.bit[nxt]
            step >>= 1

        return pos + 1


def solve() -> None:

    data = sys.stdin.read().split()
    if not data:
        return

    idx = 0
    answers = []

    while idx < len(data):
        m = int(data[idx])
        idx += 1

        bit = Fenwick(m)

        # 当前是否有一张编号 x 的券已经购买但未使用
        state = {}

        # 编号 x 上一次相关操作的位置
        last = {}

        error_line = -1

        for line_no in range(1, m + 1):
            op = data[idx]
            idx += 1

            if op == "?" or op == "？":
                if error_line == -1:
                    bit.add(line_no, 1)
                continue

            x = int(data[idx])
            idx += 1

            if error_line != -1:
                continue

            cur_state = state.get(x, 0)
            last_pos = last.get(x, 0)

            # 查询 last_pos 之后是否还有可用的 ?
            def use_question_after(pos: int) -> bool:
                before = bit.sum(pos)
                total = bit.sum(m)

                if total - before <= 0:
                    return False

                q_pos = bit.kth(before + 1)
                bit.add(q_pos, -1)
                return True

            if op == "I":
                if cur_state == 0:
                    # 当前没有未使用的 x，可以直接购买
                    state[x] = 1
                    last[x] = line_no
                else:
                    # 已经买过但没用，必须先用一个 ? 补成 O x
                    if not use_question_after(last_pos):
                        error_line = line_no
                    else:
                        state[x] = 1
                        last[x] = line_no

            else:  # op == "O"
                if cur_state == 1:
                    # 已经购买，可以直接使用
                    state[x] = 0
                    last[x] = line_no
                else:
                    # 没有可用的 x，必须用一个 ? 补成 I x
                    if not use_question_after(last_pos):
                        error_line = line_no
                    else:
                        state[x] = 0
                        last[x] = line_no

        answers.append(str(error_line))

    sys.stdout.write("\n".join(answers))


if __name__ == "__main__":
    solve()


## E 任意点

In [ ]:
## add your code here
import sys

class DSU:
    """
    并查集，用来维护哪些点属于同一个连通块。

    如果两个点能够互相到达，就把它们合并到同一个集合中。最后统计一共有多少个集合，也就是多少个连通块。
    """

    def __init__(self, n):
        # parent[i] 表示第 i 个点的父节点
        # 初始时每个点的父节点都是自己，说明每个点单独是一个连通块
        self.parent = list(range(n))

    def find(self, x):
        """
        查找 x 所在集合的代表节点。

        路径压缩：如果 x 的父节点不是自己，就不断往上找根节点，并且把中间节点直接连到根节点上，提高后续查询速度。
        """
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        """
        合并 a 和 b 所在的两个集合。如果两个点在同一行或者同一列，那么它们之间可以通过直线移动到达，所以它们属于同一个连通块。
        """
        root_a = self.find(a)
        root_b = self.find(b)

        # 如果它们本来不在同一个集合中，就合并
        if root_a != root_b:
            self.parent[root_b] = root_a


def solve():
    """
# ============================================================
# 算法分析
# ============================================================
# 本题是一个“平面点连通”问题。
#
#平面上有 n 个点，从任意一个点出发，可以向东、南、西、北四个方向直线行走，直到碰到另一个点后，才可以改变方向。
#
# 换句话说：
#
#   如果两个点在同一行，也就是 y 坐标相同，那么可以从其中一个点沿水平方向走到另一个点。
#   如果两个点在同一列，也就是 x 坐标相同，那么可以从其中一个点沿竖直方向走到另一个点。
#
# 因此，本题可以转化成图论问题：
#
#   每个点看成图中的一个节点；如果两个点的 x 坐标相同，或者 y 坐标相同，就说明它们之间可以直接到达，可以看作有一条边。
#
# 题目要求至少添加多少个点，使得任意两个点之间都可以互相到达。也就是说，最终要让所有点属于同一个连通块。
#
# ------------------------------------------------------------
# 一、为什么使用并查集
# ------------------------------------------------------------
# 并查集适合用来维护“哪些点属于同一个连通块”。
#
# 在本题中：
#
#   如果点 i 和点 j 的 x 坐标相同，说明它们在同一列，可以直接互相到达，所以把它们合并到同一个集合中。
#   如果点 i 和点 j 的 y 坐标相同，说明它们在同一行，也可以直接互相到达，所以也把它们合并到同一个集合中。
#
# 最后，并查集中有多少个不同的根节点，就说明当前这些点被分成了多少个互不连通的部分。
#
# 这些互不连通的部分就叫“连通块”。
#
# ------------------------------------------------------------
# 二、算法具体步骤
# ------------------------------------------------------------
# 第一步：
#   读入 n 个点的坐标。
#
# 第二步：
#   初始化并查集。初始时，每个点都单独属于一个集合，也就是每个点自己是一个连通块。
#
# 第三步：
#   两两比较所有点。
#
#   对于任意两个点 i 和 j：
#
#       如果 x_i == x_j，说明两个点在同一列，可以互相到达。
#       如果 y_i == y_j，说明两个点在同一行，也可以互相到达。
#       只要满足其中一个条件，就调用 union(i, j)，把它们合并到同一个连通块中。
#
# 第四步：
#   遍历所有点，使用 find(i) 找到每个点所在连通块的代表节点。把所有不同的代表节点放进 set 集合中。set 的大小就是连通块数量，记为 cnt。
#
# 第五步：
#   输出 cnt - 1。
#
# ------------------------------------------------------------
# 三、为什么答案是 cnt - 1
# ------------------------------------------------------------
# 假设当前有 cnt 个连通块。
#
# 如果 cnt = 1：说明所有点本来就已经互相连通，不需要添加任何点，答案是 0。
# 如果 cnt = 2：说明有两个互不连通的部分。
#
#   我们可以从第一个连通块中任选一个点 A，从第二个连通块中任选一个点 B。
#
#   假设：
#
#       A = (x1, y1)
#       B = (x2, y2)
#
#   那么添加一个新点：
#
#       C = (x1, y2)
#
#   新点 C 和 A 的 x 坐标相同，所以 C 能和 A 所在连通块连通；新点 C 和 B 的 y 坐标相同，所以 C 也能和 B 所在连通块连通。
#   这样，一个新点就可以把两个连通块连接起来。
#
# 如果 cnt = 3：
#
#   可以先添加一个点连接第 1 个和第 2 个连通块；再添加一个点连接合并后的连通块和第 3 个连通块。所以需要 2 个点。
#
# 推广可得：如果有 cnt 个连通块，最少需要添加 cnt - 1 个点，才能把它们全部连接成一个整体。
#
# ------------------------------------------------------------
# 四、为什么不能少于 cnt - 1
# ------------------------------------------------------------
# 一个新添加的点最多可以把两个原本不同的连通块连接起来。
#
# 因为一个新点只有一个 x 坐标和一个 y 坐标：
#
#   它可以通过相同 x 坐标连接一个连通块；也可以通过相同 y 坐标连接另一个连通块。
#
# 所以一个新点最多让连通块数量减少 1。
#
# 如果原来有 cnt 个连通块，要把它们变成 1 个连通块，至少需要减少 cnt - 1 次。因此至少需要 cnt - 1 个新点。
#
# 上面已经说明 cnt - 1 个点一定可以做到，
# 所以答案就是：cnt - 1
#
# ------------------------------------------------------------
# 五、时间复杂度分析
# ------------------------------------------------------------
# 设点的数量为 n。
#
# 代码中使用两层循环两两比较所有点：
#
#   for i in range(n):
#       for j in range(i + 1, n):
#
# 需要比较的点对数量大约是：n(n - 1) / 2
#
# 所以两两比较的时间复杂度是：O(n^2)
#
# 每次如果发现两个点在同一行或同一列，就进行一次并查集 union 操作。并查集使用了路径压缩，所以单次 find / union 操作非常接近 O(1)。
#
# 因此整体时间复杂度主要由两两比较决定：O(n^2)。题目中 n <= 100，所以 O(n^2) 的做法完全可以接受。

    """

    data = sys.stdin.read().split()

    # 如果没有输入，直接结束
    if not data:
        return

    n = int(data[0])

    points = []
    index = 1

    # 读取 n 个点的坐标
    for _ in range(n):
        x = int(data[index])
        y = int(data[index + 1])
        points.append((x, y))
        index += 2

    # 创建并查集
    dsu = DSU(n)

    # 两两比较所有点
    for i in range(n):
        x1, y1 = points[i]

        for j in range(i + 1, n):
            x2, y2 = points[j]

            # 如果两个点在同一列，或者在同一行
            # 那么它们可以互相到达，需要合并
            if x1 == x2 or y1 == y2:
                dsu.union(i, j)

    # 用集合统计不同的根节点数量
    # 每一个不同的根节点代表一个连通块
    components = set()

    for i in range(n):
        components.add(dsu.find(i))

    # 连通块数量
    cnt = len(components)

    # 最少添加的点数 = 连通块数量 - 1
    print(cnt - 1)


if __name__ == "__main__":
    solve()

## F 通配符匹配

In [ ]:
## add your code here
## 本题在解题思路上面参考了大模型的做法
import sys
import re

# ============================================================
# 算法分析
# ============================================================
# 本题是文件名通配符匹配问题。
#
# 通配符规则：
#
#   ? ：可以匹配任意一个字符；
#   * ：可以匹配任意长度的字符串，包括空字符串；
#   普通小写字母：必须和文件名中的对应字符完全相同。
#
# ------------------------------------------------------------
# 一、算法核心思想
# ------------------------------------------------------------
# 由于 '*' 可以匹配任意长度字符串，所以直接逐字符匹配比较困难。
#
# 本算法先将模式串 pattern 按照 '*' 分割成若干个片段。
#
# 例如：pattern = "ab*cd?ef*gh"
#
# 按照 '*' 分割后得到：
#
#   "ab"
#   "cd?ef"
#   "gh"
#
# 这些片段内部不再包含 '*'，只包含普通字母和 '?'。
#
# 因此，匹配整个 pattern 就变成：这些片段是否能按照顺序出现在文件名中。
#
# ------------------------------------------------------------
# 二、匹配规则
# ------------------------------------------------------------
# 如果 pattern 中没有 '*'：
#
#   文件名长度必须和 pattern 长度完全相同；
#   然后逐位判断：普通字母必须相同；? 可以匹配任意字符。
#
# 如果 pattern 中有 '*'：
#
#   1. 如果 pattern 不是以 '*' 开头，那么第一段必须匹配文件名开头。
#
#   2. 如果 pattern 不是以 '*' 结尾，那么最后一段必须匹配文件名结尾。
#
#   3. 中间被 '*' 分隔开的片段，只需要按顺序出现在剩余文件名区间中即可。
#
# 例如：pattern = "ab*cd*ef"
#
# 文件名必须：
#
#   以 "ab" 开头；
#   以 "ef" 结尾；
#   中间某处按顺序出现 "cd"。
#
# ------------------------------------------------------------
# 三、如何处理每个片段
# ------------------------------------------------------------
# 对于一个不含 '*' 的片段：
#
#   普通字母必须精确匹配；? 的位置不用检查，因为它可以匹配任意字符。
#
# 例如：片段 "a?c"
#
# 可以匹配：
#
#   "abc"
#   "aac"
#   "azc"
#
# 但不能匹配："abb"
#
# 代码中将每个片段封装成 Segment 对象，并把其中连续的普通字母块保存下来。这样检查片段时，只需要检查普通字母块是否匹配，不需要逐个处理所有 ?。
#
# ------------------------------------------------------------
# 四、为什么要优化查找
# ------------------------------------------------------------
# 如果中间片段很长，并且包含很多 ?，直接从文件名每个位置开始尝试匹配会比较慢。所以代码会从片段中选择一个普通字母块作为“锚点”。
#
# 例如：片段是 "aaa?b"
#
# 如果文件名中 'a' 很多，'b' 很少，那么优先查找 'b' 的位置会更快。找到锚点后，再反推出整个片段可能的起始位置，最后检查整个片段是否匹配。
#
# ------------------------------------------------------------
# 五、整体执行步骤
# ------------------------------------------------------------
# 1. 读入通配符 pattern。
#
# 2. 将 pattern 按 '*' 分割成多个 Segment 片段。
#
# 3. 对每一个文件名 filename：
#
#    如果 pattern 中没有 '*'：判断 filename 长度是否等于 pattern 长度；再检查该片段是否完全匹配。
#    如果 pattern 中有 '*'：先处理必须匹配开头的片段；再处理必须匹配结尾的片段；最后在中间剩余范围内按顺序查找每个片段。
#
# 4. 如果所有片段都能成功匹配，则输出 YES。
#
# 5. 否则输出 NO。
#
# ------------------------------------------------------------
# 六、时间复杂度分析
# ------------------------------------------------------------
# 设：
#
#   L 表示文件名长度；
#   P 表示通配符长度；
#   n 表示文件数量。
#
# 对每个文件名，算法需要按照顺序查找 pattern 分割出的片段。
#
# 普通情况下，每个片段查找和验证都在文件名范围内完成，因此单个文件名的匹配时间大致与文件名长度和模式串长度相关。
#
# 可以近似理解为：O(L + P)
#
# 对 n 个文件名分别匹配，所以总时间复杂度约为：O(n * (L + P))
#
# 代码中还针对短片段使用正则、针对纯普通字符串使用 bytes.find、针对复杂片段选择稀有字符块作为锚点，这些优化可以减少实际运行时间。

QUESTION = ord('?')
STAR = ord('*')

# 对很短的片段，用 Python 底层的正则匹配，速度更快。这里的正则不会包含 *，所以不会出现复杂回溯
SMALL_RE_LIMIT = 64


class Segment:


    __slots__ = ("length", "runs", "fast_re", "is_plain")

    def __init__(self, raw):
        """
        raw 是 bytes 类型的一段模式串。
        """

        # 当前片段的长度
        self.length = len(raw)

        # runs 用来保存当前片段中的连续普通字母块
        #
        # 例如：raw = b"ab??cde?fg"
        #
        # 普通字母块有：
        #   b"ab"   起始位置 0
        #   b"cde"  起始位置 4
        #   b"fg"   起始位置 8
        #
        # runs 中保存：(起始位置, 普通字母块, 该字母块中出现过的字符集合)
        self.runs = []

        has_question = False
        i = 0

        while i < len(raw):
            if raw[i] == QUESTION:
                has_question = True
                i += 1
            else:
                start = i

                # 找到一整段连续普通小写字母
                while i < len(raw) and raw[i] != QUESTION:
                    i += 1

                literal = raw[start:i]

                # set(literal) 用于后面选择“更稀有”的匹配锚点
                chars = tuple(set(literal))

                self.runs.append((start, literal, chars))

        # 判断这一段是不是纯普通字符串
        #
        # 例如：
        #   "abcdef" 是纯普通字符串
        #   "abc?def" 不是
        self.is_plain = (
            len(self.runs) == 1
            and self.runs[0][0] == 0
            and len(self.runs[0][1]) == self.length
        )

        # 如果片段比较短，并且里面有 '?'，就编译成一个固定长度正则。
        #
        # 注意：这里不会有 '*'，只有普通字母和 '.' ，所以不会产生复杂的回溯问题。
        if has_question and self.length <= SMALL_RE_LIMIT:
            buf = bytearray()

            for ch in raw:
                if ch == QUESTION:
                    # '?' 可以匹配任意一个字符
                    buf.extend(b'.')
                else:
                    # 题目说文件名只包含小写字母，所以普通字母可以直接放进去
                    buf.append(ch)

            self.fast_re = re.compile(bytes(buf))
        else:
            self.fast_re = None


class WildcardMatcher:
    """
    通配符匹配器。

    支持：
        '?' 匹配任意一个字符
        '*' 匹配任意长度字符串，包括空字符串
    """

    def __init__(self, pattern):
        self.has_star = STAR in pattern

        # 判断 pattern 是否以 '*' 开头
        self.leading_star = pattern.startswith(b'*')

        # 判断 pattern 是否以 '*' 结尾
        self.trailing_star = pattern.endswith(b'*')

        # 按 '*' 切分模式串
        self.segments = [Segment(part) for part in pattern.split(b'*')]

    def check_at(self, s, seg, base):
        """
        判断片段 seg 能不能从文件名 s 的 base 位置开始匹配。

        例如：
            s = b"xxacatctc"
            seg = "aca?ctc"
            base = 2

        那么需要判断：
            s[2:9] 是否能匹配 "aca?ctc"
        """

        # 长度越界，肯定不匹配
        if base < 0 or base + seg.length > len(s):
            return False

        # 如果这个片段很短并且已经编译了正则，就直接用正则判断
        if seg.fast_re is not None:
            m = seg.fast_re.match(s, base, base + seg.length)
            return m is not None and m.end() == base + seg.length

        # 否则只检查普通字母块
        # '?' 位置不用检查，因为它可以匹配任意字符
        for offset, literal, _ in seg.runs:
            if not s.startswith(literal, base + offset):
                return False

        return True

    def calc_freq(self, s):
        """
        统计文件名中每个字符出现的次数。后面在寻找片段时，会优先选择“出现次数少”的字符块作为锚点，这样可以避免从头到尾大量无效枚举。
        """

        freq = [0] * 256

        for ch in s:
            freq[ch] += 1

        return freq

    def choose_anchor(self, seg, freq):
        """
        为一个片段选择最适合作为查找入口的普通字母块。

        为什么要选锚点？

        例如：片段是 "aaa?b"。如果文件名中 'a' 特别多，而 'b' 很少，那我们应该优先找 'b'，而不是从每一个 'a' 开始尝试。这样可以大幅减少循环次数。
        """

        best = None

        for offset, literal, chars in seg.runs:
            # 当前字母块中，出现次数最少的字符
            rare_count = min(freq[ch] for ch in chars)

            # 排序规则：
            #   1. rare_count 越小越好
            #   2. 如果 rare_count 一样，literal 越长越好
            candidate = (rare_count, -len(literal), offset, literal)

            if best is None or candidate < best:
                best = candidate

        return best[2], best[3], best[0]

    def find_segment(self, s, seg, start, end, freq):
        """
        在 s 的区间 [start, end) 中，查找片段 seg 最早可以匹配的位置。

        返回：如果找到，返回起始位置；如果找不到，返回 -1。
        """

        # 如果剩余长度不够放下这个片段，直接失败
        if start + seg.length > end:
            return -1

        # 空片段直接匹配成功
        if seg.length == 0:
            return start

        # 如果这个片段全是 '?'，那么它可以匹配任意同长度字符串，最早位置就是 start
        if not seg.runs:
            return start

        # 如果这个片段比较短，并且已经有正则，直接用底层 C 实现的 search，速度比 Python 手写循环快
        if seg.fast_re is not None:
            m = seg.fast_re.search(s, start, end)

            if m is None:
                return -1

            return m.start()

        # 如果这个片段是纯普通字符串，直接用 bytes.find，底层速度很快
        if seg.is_plain:
            return s.find(seg.runs[0][1], start, end)

        # 对较长且含 '?' 的片段，选择一个合适的普通字母块作为锚点
        anchor_offset, anchor_text, rare_count = self.choose_anchor(seg, freq)

        # 如果锚点中的某个字符在文件名里根本不存在，那这个片段一定无法匹配
        if rare_count == 0:
            return -1

        # 片段起点 base 的最大值
        max_base = end - seg.length

        # anchor_text 在文件名中的位置 occ 满足：occ = base + anchor_offset
        #
        # 所以：base = occ - anchor_offset
        search_from = start + anchor_offset
        search_end = max_base + anchor_offset + len(anchor_text)

        while True:
            occ = s.find(anchor_text, search_from, search_end)

            if occ == -1:
                return -1

            base = occ - anchor_offset

            # 找到锚点后，还要检查整个片段是否匹配
            if self.check_at(s, seg, base):
                return base

            # 当前这个位置不行，继续找下一个锚点出现位置
            search_from = occ + 1

    def match(self, s):
        """
        判断文件名 s 是否能被整个通配符模式匹配。
        """

        # 情况一：
        # pattern 中没有 '*'
        #
        # 那么文件名长度必须和 pattern 长度完全相同，然后只需要判断 '?' 和普通字母是否匹配即可。
        if not self.has_star:
            seg = self.segments[0]
            return len(s) == seg.length and self.check_at(s, seg, 0)

        # left 和 right 表示当前还可以匹配的文件名范围：[left, right)
        left = 0
        right = len(s)

        # first 和 last 表示中间还需要按顺序查找哪些片段
        first = 0
        last = len(self.segments) - 1

        # 如果 pattern 不是以 '*' 开头，那么第一段必须严格匹配文件名开头。
        #
        # 例如：pattern = "abc*def"
        #文件名必须以 abc 开头。

        if not self.leading_star:
            first_seg = self.segments[0]

            if not self.check_at(s, first_seg, 0):
                return False

            left = first_seg.length
            first = 1

        # 如果 pattern 不是以 '*' 结尾，那么最后一段必须严格匹配文件名结尾。
        #
        # 例如：pattern = "abc*def"。文件名必须以 def 结尾。
        if not self.trailing_star:
            last_seg = self.segments[-1]
            base = right - last_seg.length

            if not self.check_at(s, last_seg, base):
                return False

            right = base
            last -= 1

        # 如果前缀和后缀已经互相冲突，说明匹配失败
        if left > right:
            return False

        # freq 不一定每个文件都需要算
        # 只有遇到较长且带 '?' 的中间片段时才需要
        freq = None

        # 处理中间片段
        #
        # 因为这些片段之间原来有 '*'，所以它们只需要按照顺序出现在剩余区间中即可。
        for idx in range(first, last + 1):
            seg = self.segments[idx]

            # 空片段直接跳过，例如 "ab**cd" 中间会产生空片段
            if seg.length == 0:
                continue

            # 如果这一段全是 '?'，它可以直接吃掉固定长度的字符
            if not seg.runs:
                if left + seg.length > right:
                    return False

                left += seg.length
                continue

            # 只有较长、含 '?'、不能直接正则处理、也不是纯普通字符串时，才需要字符频率来选择锚点
            need_freq = (seg.fast_re is None and not seg.is_plain)

            if need_freq and freq is None:
                freq = self.calc_freq(s)

            pos = self.find_segment(s, seg, left, right, freq)

            if pos == -1:
                return False

            # 当前片段匹配完之后，后面的片段只能从它后面继续找
            left = pos + seg.length

            if left > right:
                return False

        return True


def solve():
    """
    输入：
        第一行：通配符字符串
        第二行：文件数量 n
        接下来 n 行：文件名

    输出：
        每个文件名对应输出 YES 或 NO
    """

    data = sys.stdin.buffer.read().split()

    if not data:
        return

    pattern = data[0]
    n = int(data[1])

    matcher = WildcardMatcher(pattern)

    ans = []

    for i in range(n):
        filename = data[2 + i]

        if matcher.match(filename):
            ans.append("YES")
        else:
            ans.append("NO")

    sys.stdout.write("\n".join(ans))


if __name__ == "__main__":
    solve()

## G 汉诺塔

In [ ]:
## add your code here
import sys


def solve():
    """
    汉诺塔变形题解法说明：

    一、为什么不能直接模拟？
    --------------------------------
    题目中 n 最大是 30，汉诺塔的移动次数可能达到指数级。如果我们一步一步模拟，可能要模拟几亿甚至更多步，肯定会超时。

    所以我们要用“递推”的方法，只计算移动次数，不真的把每一步都模拟出来。

    二、我们定义两个数组
    --------------------------------
    cnt[k][S] 表示：k 个盘子一开始全部在柱子 S 上，按照题目给定的优先级贪心移动，第一次把这 k 个盘子全部移动到另一根柱子上，需要多少步。

    to[k][S] 表示：上面这个过程结束后，k 个盘子最终在哪根柱子上。

    例如：
        cnt[3]['A'] = 7
        to[3]['A'] = 'B'

    表示 3 个盘子从 A 出发，第一次全部移动到另一根柱子时，用了 7 步，最终到了 B。

    三、递推思路
    --------------------------------
    假设我们要求 k 个盘子从柱子 S 出发的情况。

    最大的盘子是第 k 个盘子。它一开始在最下面，所以必须先把上面的 k-1 个小盘子移走。

    第一步：
        先把 k-1 个小盘子从 S 移到另一根柱子 small 上。这个过程需要 cnt[k-1][S] 步。small = to[k-1][S]

    第二步：
        此时第 k 个大盘子露出来了。由于小盘子都在 small 上，所以大盘子只能移动到第三根柱子 big 上。这一步需要 1 步。

    第三步：
        现在局面变成：
            k-1 个小盘子在 small 上，
            第 k 个大盘子在 big 上。

        接下来继续按照同样的贪心规则移动 k-1 个小盘子。如果小盘子最终移动到了 big 上，那么 k 个盘子就全部在 big 上，过程结束。

        如果小盘子没有移动到 big 上，那么第 k 个大盘子还要继续移动一次，然后重复这个过程。

    因为柱子只有 A、B、C 三根，所以这个“宏观过程”非常短，不需要真的模拟每一个盘子的移动。
    """

    data = sys.stdin.read().split()
    if not data:
        return

    n = int(data[0])

    # priority 存储 6 种操作的优先级顺序，例如：["AB", "BC", "CA", "BA", "CB", "AC"]
    priority = data[1:]

    pegs = ['A', 'B', 'C']

    # rank[move] 表示某个操作的优先级排名，数字越小，优先级越高。例如 priority[0] 是优先级最高的操作
    rank = {}
    for i, move in enumerate(priority):
        rank[move] = i

    # cnt[k][S]：k 个盘子从 S 出发，第一次整体到另一根柱子所需步数
    # to[k][S]：k 个盘子从 S 出发，第一次整体到达的目标柱子
    cnt = [{} for _ in range(n + 1)]
    to = [{} for _ in range(n + 1)]

    def third_peg(x, y):
        """
        返回除了 x 和 y 以外的第三根柱子。

        例如：
            x = 'A', y = 'B'
            那么第三根柱子就是 'C'
        """
        for p in pegs:
            if p != x and p != y:
                return p

    # 处理 k = 1 的基础情况，只有 1 个盘子时，它可以从 S 移到另外两根柱子之一。按照优先级，选择优先级最高的合法移动即可。
    for s in pegs:
        best_move = None

        for t in pegs:
            if t == s:
                continue

            move = s + t

            if best_move is None or rank[move] < rank[best_move]:
                best_move = move

        cnt[1][s] = 1
        to[1][s] = best_move[1]

    # 从 2 个盘子开始递推，一直算到 n 个盘子
    for k in range(2, n + 1):
        for start in pegs:
            total_steps = 0

            # 先移动上面的 k-1 个小盘子
            small = to[k - 1][start]
            total_steps += cnt[k - 1][start]

            # 第 k 个大盘子原来在 start 上
            big = start

            # 小盘子在 small 上，大盘子只能移动到第三根柱子
            big = third_peg(big, small)
            total_steps += 1

            # 现在：
            #   k-1 个小盘子在 small 上
            #   第 k 个大盘子在 big 上
            #
            # 接下来重复移动小盘子和大盘子的过程，直到小盘子移动到 big 上，形成完整的 k 个盘子塔。
            while True:
                # 把 k-1 个小盘子从 small 按照规则移动到另一根柱子
                next_small = to[k - 1][small]
                total_steps += cnt[k - 1][small]
                small = next_small

                # 如果小盘子移动到了大盘子所在的柱子 big，那么 k 个盘子就全部叠在 big 上，过程结束。
                if small == big:
                    cnt[k][start] = total_steps
                    to[k][start] = big
                    break

                # 否则，小盘子没有到 big 上。此时大盘子又可以移动一次，它只能移动到除了 small 和 big 之外的第三根柱子。
                big = third_peg(big, small)
                total_steps += 1

    # 题目要求：所有盘子从 A 移动到 B 或 C 所需步数
    print(cnt[n]['A'])


if __name__ == "__main__":
    solve()

## H 马步距离

In [ ]:
import sys


def knight_min_steps(x1, y1, x2, y2):
    """
    计算国际象棋/中国象棋中，“马”从 (x1, y1) 到 (x2, y2) 的最少移动次数。

    马每一步可以走：
        横向 1 格 + 纵向 2 格  或者  横向 2 格 + 纵向 1 格

    例如：
        (x, y) 可以走到：
        (x + 1, y + 2)
        (x + 2, y + 1)
        (x + 1, y - 2)
        (x + 2, y - 1)
        (x - 1, y + 2)
        (x - 2, y + 1)
        (x - 1, y - 2)
        (x - 2, y - 1)

    因为棋盘无限大，所以不用担心越界问题。
    """

    # 第一步：先计算两个点在横坐标和纵坐标上的差值
    # abs 表示取绝对值，因为从左往右走和从右往左走，本质是一样的
    dx = abs(x2 - x1)
    dy = abs(y2 - y1)

    # 第二步：为了方便后面统一计算，让 dx 表示较大的那个差值
    # 例如 dx=3, dy=7，我们就交换成 dx=7, dy=3
    if dx < dy:
        dx, dy = dy, dx

    # 第三步：处理最简单的情况。如果起点和终点本来就是同一个点，那么不需要移动
    if dx == 0 and dy == 0:
        return 0

    # 第四步：处理两个特殊情况
    #
    # 情况1：从 (0,0) 到 (1,0)。虽然看起来只差一格，但马不能直接走过去。最少需要 3 步
    #
    # 一种走法：
    # (0,0) -> (2,1) -> (0,2) -> (1,0)
    if dx == 1 and dy == 0:
        return 3

    # 情况2：从 (0,0) 到 (2,2)。按照一般公式会算成 2 步，但实际上 2 步无法到达。最少需要 4 步
    if dx == 2 and dy == 2:
        return 4

    # 第五步：使用通用公式计算大多数情况
    #
    # 为什么需要下面两个值？
    #
    # 1. (dx + 1) // 2
    #    因为马一步最多让横向距离减少 2
    #    所以横向差值 dx 至少需要 ceil(dx / 2) 步
    #
    #    在 Python 中：
    #    ceil(dx / 2) 可以写成 (dx + 1) // 2
    #
    # 2. (dx + dy + 2) // 3
    #    因为马每走一步，总共可以在横纵方向上移动 3 格
    #    所以总距离 dx + dy 至少需要 ceil((dx + dy) / 3) 步
    #
    #    在 Python 中：
    #    ceil((dx + dy) / 3) 可以写成 (dx + dy + 2) // 3
    #
    # 最少步数一定不能小于这两个限制，所以取它们的最大值
    ans = max((dx + 1) // 2, (dx + dy + 2) // 3)

    # 第六步：调整奇偶性
    #
    # 马每走一步，横坐标和纵坐标变化量之和一定是奇数：
    # 例如：
    #   1 + 2 = 3
    #   2 + 1 = 3
    #
    # 所以马每走一步，棋盘格子的颜色会发生变化。
    #
    # 这就导致：如果需要到达的位置 dx + dy 和当前步数 ans 的奇偶性不匹配，那么 ans 还要再加 1。
    #
    # 判断方法：如果 ans + dx + dy 是奇数，说明奇偶性不匹配。
    if (ans + dx + dy) % 2 == 1:
        ans += 1

    # 返回最终答案
    return ans


def solve():
    """
    主函数：负责读取输入，调用计算函数，然后输出答案。
    """

    # 读取所有输入数据
    # 题目输入只有 4 个整数：xp yp xs ys
    data = sys.stdin.read().split()

    # 如果没有输入，就直接结束
    if not data:
        return

    # 把输入的字符串转换成整数
    xp, yp, xs, ys = map(int, data[:4])

    # 调用函数，计算最少马步移动次数
    result = knight_min_steps(xp, yp, xs, ys)

    # 输出答案
    print(result)


# Python 程序入口
if __name__ == "__main__":
    solve()

## I 直方图最大矩形

In [ ]:
## add your code here
#本题在算法分析部分借鉴了一下大模型的思路
# ============================================================
# 算法分析
# ============================================================
# 本题是“柱状图中最大矩形”问题。
#
# 给定一个数组 heights，其中 heights[i] 表示第 i 根柱子的高度，每根柱子的宽度都为 1，并且所有柱子都是相邻的。
#
# 目标是：找到若干根连续柱子组成的最大矩形面积。
#
# 矩形的面积计算方式为：面积 = 高度 × 宽度
#
# 其中：高度由选中连续柱子中的最矮柱子决定；宽度由连续柱子的数量决定。
#
# 例如：heights = [3, 4, 7, 8, 1, 2]
#
# 如果选择高度为 7 和 8 的两根柱子，那么矩形高度只能取较矮的 7，宽度为 2，所以面积为：7 × 2 = 14
#
# 因此该例子的最大矩形面积是 14。
#
# ------------------------------------------------------------
# 一、为什么不能直接暴力枚举
# ------------------------------------------------------------
# 最直观的方法是枚举所有连续区间。
#
# 例如：枚举第 1 根到第 1 根；枚举第 1 根到第 2 根；枚举第 1 根到第 3 根；...
#
# 对每一个区间，都找出其中最矮的柱子作为矩形高度。
#
# 但是这样会非常慢。因为数组长度最大可以达到：10^5
#
# 如果枚举所有区间，时间复杂度至少是 O(n^2)，数据量大时会超时。所以需要更高效的方法。
#
# ------------------------------------------------------------
# 二、算法核心思想：单调栈
# ------------------------------------------------------------
# 本题可以使用“单调栈”解决。
#
# 栈中保存的是柱子的下标，而不是柱子的高度。
#
# 为什么保存下标？
#
# 因为计算矩形面积时，不仅需要知道高度，还需要知道这个高度能够向左、向右扩展多宽。宽度需要通过下标相减计算出来。
#
# ------------------------------------------------------------
# 三、单调栈维护什么性质
# ------------------------------------------------------------
# 本算法维护一个“高度单调递增”的栈。也就是说，栈中下标对应的柱子高度是从低到高排列的。
#
# 当当前柱子高度大于等于栈顶柱子高度时，说明当前柱子不会截断栈顶柱子的右边界，可以直接入栈。
#
# 当当前柱子高度小于栈顶柱子高度时，说明栈顶柱子不能继续向右扩展了。
#
# 此时就可以确定：栈顶柱子的右边第一个比它矮的柱子，就是当前柱子。所以应该弹出栈顶柱子，并以它作为矩形高度计算面积。
#
# ------------------------------------------------------------
# 四、为什么弹出时可以计算面积
# ------------------------------------------------------------
# 假设被弹出的柱子下标为 mid。
#
# 那么：height = heights[mid]  表示当前矩形的高度。
#
# 当前扫描到的位置 i，是 mid 右边第一个比它矮的位置。因此右边界是：right = i
#
# 弹出 mid 以后，新的栈顶 stack[-1] 是 mid 左边第一个比它矮的位置。因此左边界是：left = stack[-1]
#
# 那么以 heights[mid] 为高度的矩形，可以扩展的范围是：left + 1 到 right - 1
#
# 所以宽度为：width = right - left - 1；   面积为：area = heights[mid] × width
#
# 每弹出一个柱子，就计算一次以该柱子为高度时能形成的最大矩形面积。
#
# ------------------------------------------------------------
# 五、为什么要在左右两边加 0
# ------------------------------------------------------------
# 原数组两边加 0：heights = [0] + heights + [0]
#
# 左边加 0 的作用：保证栈底永远有一个比所有柱子都矮的柱子，这样计算左边界时，不容易出现栈为空的问题。
#
# 右边加 0 的作用：当遍历到最后时，这个 0 会比所有真实柱子都矮，从而强制把栈中剩余的柱子全部弹出并计算面积。
#
# 如果右边不加 0，可能有些递增柱子一直留在栈里，没有机会被计算。
#
# 六、算法步骤
# ------------------------------------------------------------
# 1. 如果 heights 为空，直接返回 0。
#
# 2. 在 heights 左右两边各加一个高度为 0 的柱子。
#
# 3. 创建一个空栈 stack，用来保存柱子的下标。
#
# 4. 从左到右遍历每根柱子。
#
# 5. 如果当前柱子高度小于栈顶柱子高度，说明栈顶柱子右边界已经确定，弹出栈顶并计算面积。
#
# 6. 计算面积时：
#
#       height = heights[mid]
#       width = right - left - 1
#       area = height × width
#
# 7. 不断更新最大面积 ans。
#
# 8. 遍历结束后，返回 ans。
#
# ------------------------------------------------------------
# 七、时间复杂度分析
# ------------------------------------------------------------
# 虽然代码中有 for 循环和 while 循环，但整体时间复杂度不是 O(n^2)。
#
# 原因是：每根柱子的下标最多入栈一次；每根柱子的下标最多出栈一次。
#
# 也就是说，对于 n 根柱子，总入栈次数最多是 n 次，总出栈次数最多也是 n 次。
#
# 因此所有操作加起来是线性级别的。所以时间复杂度为：O(n)

class Solution:
    def largestRectangleArea(self, heights):
        """
        给定一个柱状图 heights，求能够形成的最大矩形面积。

        参数：
            heights: List[int]
            例如：[3, 4, 7, 8, 1, 2]

        返回：
            int，最大矩形面积
        """

        # 如果数组为空，说明没有柱子，最大面积就是 0
        if not heights:
            return 0

        # --------------------------------------------------------
        # 为什么要在左右两边各加一个 0？
        #
        # 原数组：
        # [3, 4, 7, 8, 1, 2]
        #
        # 加 0 后：
        # [0, 3, 4, 7, 8, 1, 2, 0]
        #
        # 左边加 0：是为了防止栈为空，方便统一计算宽度。
        #
        # 右边加 0：是为了在最后强制把栈里剩下的柱子全部弹出来计算面积。
        #
        # 因为 0 比所有柱子都矮，所以最后一定会触发计算。
        # --------------------------------------------------------
        heights = [0] + heights + [0]

        # stack 用来存放柱子的下标
        #
        # 注意：栈里存的是“下标”，不是高度。
        #
        # 为什么存下标？因为计算矩形面积时，需要知道宽度。宽度需要通过下标相减得到。
        stack = []

        # ans 用来记录当前找到的最大矩形面积
        ans = 0

        # 开始从左到右扫描每一根柱子
        for i in range(len(heights)):

            # 当前柱子的高度
            current_height = heights[i]

            # --------------------------------------------------------
            # 维护一个“高度单调递增”的栈
            #
            # 如果当前柱子比栈顶柱子矮，说明栈顶柱子不能继续向右扩展了。此时就可以用栈顶柱子的高度来计算一次矩形面积。
            # --------------------------------------------------------
            while stack and heights[stack[-1]] > current_height:

                # 弹出栈顶柱子
                # mid 是当前要计算面积的柱子下标
                mid = stack.pop()

                # 当前矩形的高度就是 heights[mid]
                height = heights[mid]

                # 弹出 mid 之后，新的栈顶就是 mid 左边第一个比它矮的柱子
                left = stack[-1]

                # 当前 i 就是 mid 右边第一个比它矮的柱子
                right = i

                # ----------------------------------------------------
                # 为什么宽度是 right - left - 1？
                #
                # left 是左边第一个比 mid 矮的位置；right 是右边第一个比 mid 矮的位置。所以真正能形成矩形的范围是：left + 1  到  right - 1
                #
                # 宽度就是：right - left - 1
                # ----------------------------------------------------
                width = right - left - 1

                # 当前矩形面积 = 高度 × 宽度
                area = height * width

                # 更新最大面积
                ans = max(ans, area)

            # 当前柱子入栈。入栈后，栈中的柱子高度仍然保持递增
            stack.append(i)

        # 返回最大矩形面积
        return ans

## J 消防局的设立

In [ ]:
## add your code here
##本题在算法分析思路上面借助了大模型
# ============================================================
# 算法分析
# ============================================================

# 一、为什么要从叶子往根处理
# ------------------------------------------------------------
# 这是一棵树。
#
# 对于树上的覆盖问题，常用方法是从叶子节点往根节点贪心处理。
#
# 原因是：叶子节点没有子节点，它只能被自己、父节点、祖父节点上的消防局覆盖。
#
# 如果某个很深的节点一直没有被覆盖，那么越往上走，距离会越来越远。当某个未覆盖节点距离当前节点已经达到 2 时，如果此时还不建消防局，再往父节点传递，距离就会变成 3，超过消防局的覆盖范围。
#
# 所以当发现“当前节点的孙子辈或更深处已经快要超出覆盖范围”时，就必须在当前节点建消防局。
#
# 这就是本题贪心的核心。
#
# ------------------------------------------------------------
# 二、整体算法思路
# ------------------------------------------------------------
# 第一步：先根据输入建立无向树。
#
# 第二步：以 1 号基地为根，将这棵无向树整理成有根树。
#
#   代码中使用 stack 从 1 号节点开始遍历，得到：parent[x]：表示 x 的父节点；order：表示从根开始遍历得到的节点顺序。
#
# 第三步：反向遍历 order。
#
#   因为 order 是从根往下得到的顺序，所以 reversed(order) 就相当于从叶子节点往根节点处理。
#
# 第四步：
#
#   对每个节点 u，根据它的孩子节点状态，判断：
#
#       1. 是否必须在 u 建消防局；
#       2. u 是否已经被子树中的消防局覆盖；
#       3. 是否还需要把未覆盖信息继续向父节点传递。
#
# 第五步：
#
#   最后处理根节点。如果根节点这边仍然存在未覆盖点，就必须在根节点补建一个消防局。
#
# ------------------------------------------------------------
# 三、状态设计
# ------------------------------------------------------------
# 代码中主要使用两个数组：need[u]和down[u]
# ------------------------------------------------------------
# 1. need[u] 的含义
# ------------------------------------------------------------
# need[u] 用来表示 u 这棵子树是否还有未覆盖的点需要向上汇报。
#
#   need[u] = -1
#
#       表示以 u 为根的子树已经被覆盖好了，不需要父节点帮忙。
#
#   need[u] = 0
#
#       表示 u 自己还没有被覆盖。也就是说，未覆盖点距离 u 为 0。
#
#   need[u] = 1
#
#       表示 u 的某个孩子还没有被覆盖。也就是说，未覆盖点距离 u 为 1。
#
# 为什么没有 need[u] = 2？
#
#   因为如果某个未覆盖点距离当前节点已经为 2，那么就不能再往父节点传了。再往上传，距离会变成 3，已经超过消防局的覆盖范围。
#
#   所以当发现孩子传来的 need[v] = 1 时，说明 v 的孩子未覆盖，这个未覆盖点距离当前节点 u 正好为 2。此时必须在 u 建消防局。
#
# ------------------------------------------------------------
# 2. down[u] 的含义
# ------------------------------------------------------------
# down[u] 表示：
#
#   在 u 的子树中，距离 u 最近的消防局距离是多少。
#
# 具体来说：
#
#   down[u] = 0 表示消防局就在 u。
#
#   down[u] = 1 表示消防局在 u 的某个孩子节点。
#
#   down[u] = 2 表示消防局在 u 的某个孙子节点。
#
#   down[u] = INF 表示 u 往下两层之内没有消防局。
#
# 为什么要记录 down[u]？
#
#   因为判断 u 是否被覆盖时，需要知道 u 的子树里有没有距离 u 不超过 2 的消防局。
#
#   如果 down[u] <= 2，说明 u 已经被下面的消防局覆盖。
#
#   如果 down[u] > 2，说明 u 还没有被覆盖。
#
# ------------------------------------------------------------
# 四、每个节点如何决策
# ------------------------------------------------------------
# 假设当前正在处理节点 u。
#
# 我们会遍历 u 的所有孩子 v，根据孩子的状态决定 u 该怎么做。
#
# ------------------------------------------------------------
# 情况一：某个孩子 need[v] == 1
# ------------------------------------------------------------
# need[v] == 1 表示：v 的某个孩子还没有被覆盖。
#
# 也就是说，这个未覆盖点距离 v 为 1，距离当前节点 u 为 2。如果不在 u 建消防局，那么这个未覆盖点再往上传到 u 的父节点时，距离就会变成 3。
#
# 消防局只能覆盖距离不超过 2 的基地，所以距离 3 就无法覆盖了。
#
# 因此：只要存在孩子 v 满足 need[v] == 1，就必须在当前节点 u 建消防局。
#
# 代码中用：forced_build = True 表示当前节点必须建消防局。
#
# ------------------------------------------------------------
# 情况二：必须在当前节点建消防局
# ------------------------------------------------------------
# 如果 forced_build == True，那么在 u 建一个消防局。此时：answer += 1。表示消防局数量增加 1。
#
# 由于消防局就在 u，所以：down[u] = 0。并且 u 的子树中原来那些未覆盖点也被覆盖了，所以：need[u] = -1 表示 u 的子树不再需要向父节点求助。
#
# ------------------------------------------------------------
# 情况三：当前节点不必须建消防局
# ------------------------------------------------------------
# 如果 forced_build == False，说明当前还没有出现“距离 u 为 2 的未覆盖点”。
#
# 这时需要判断：
#
#   1. u 自己是否已经被子树中的消防局覆盖；
#   2. u 的孩子中是否还有未覆盖点。
#
# 代码中先计算：min_down = min(down[v] + 1)  其中 v 是 u 的孩子。
#
# down[v] 表示 v 子树中最近消防局到 v 的距离，那么这个消防局到 u 的距离就是：down[v] + 1
#
# 所以 min_down 表示：u 的孩子子树里，距离 u 最近的消防局距离。
#
# 如果 min_down <= 2，说明 u 可以被下面的消防局覆盖。
#
# 如果 min_down > 2，说明 u 往下两层内没有消防局能覆盖它，此时 u 自己还没有被覆盖。
#
# ------------------------------------------------------------
# 情况四：孩子自己没有被覆盖
# ------------------------------------------------------------
# 如果某个孩子 v 满足：need[v] == 0  说明孩子 v 自己还没有被覆盖。这个未覆盖点距离当前节点 u 为 1。
#
# 如果 u 的某个孩子位置上已经有消防局，也就是：down[u] == 1  那么这个消防局可以覆盖 u 的其他孩子。
#
# 因为两个孩子之间的距离是：child1 -> u -> child2  距离为 2。
#
# 所以当 down[u] == 1 时，孩子未覆盖的问题可以被这个孩子上的消防局解决。
#
# 否则，如果 down[u] != 1，说明没有合适的消防局覆盖这个未覆盖孩子，那么需要把这个信息向父节点传递。
#
# 此时设置：need[u] = 1 表示：u 的某个孩子还没有被覆盖。
#
# ------------------------------------------------------------
# 五、为什么最后要单独处理根节点
# ------------------------------------------------------------
# 普通节点如果还有未覆盖点，可以把信息继续传给父节点。但是根节点没有父节点。
#
# 所以如果所有节点处理完后：need[1] >= 0 说明根节点或根节点的孩子仍然存在未覆盖情况。这时只能在根节点再建一个消防局。
#
# 因此：answer += 1
#
# ------------------------------------------------------------
# 六、为什么这个贪心是正确的
# ------------------------------------------------------------
# 本题的关键在于：消防局的覆盖半径只有 2。
#
# 当某个未覆盖点距离当前节点 u 已经为 2 时，如果还不在 u 建消防局，那么这个未覆盖点传到 u 的父节点时距离会变成 3。
#
# 距离 3 已经无法被父节点上的消防局覆盖。
#
# 因此，在这种情况下，当前节点 u 是最后一个能够覆盖该未覆盖点的安全位置，必须在 u 建消防局。
#
# 这就是代码中：
#
#   if need[v] == 1:
#       forced_build = True
#
# 的原因。
#
# 这种从叶子往根的处理方式保证：
#
#   1. 不会过早建消防局；
#   2. 只有当不建就会导致覆盖失败时才建；
#   3. 每次建消防局都尽量向上放，从而覆盖更多节点。
#
# 因此可以得到最少的消防局数量。
#
# ------------------------------------------------------------
# 七、时间复杂度分析
# ------------------------------------------------------------
# 设基地数量为 n。
#
# 建图时，需要处理 n - 1 条边，时间复杂度为：O(n)
#
# 第一次用栈遍历整棵树，确定 parent 和 order，每个节点访问一次，每条边最多检查两次，时间复杂度为：O(n)
#
# 第二次从叶子往根处理，每个节点访问一次，每条边也只会被检查常数次，时间复杂度为：O(n)
#
# 因此总时间复杂度为：O(n)
import sys

def solve():

    data = sys.stdin.buffer.read().split()

    # 如果没有输入，直接结束
    if not data:
        return

    n = int(data[0])

    # 特殊情况：如果只有 1 个基地，那么只需要建 1 个消防局
    if n == 1:
        print(1)
        return

    # 建图：因为道路是双向的，所以用邻接表存储
    # adj[x] 表示和基地 x 直接相连的所有基地
    adj = [[] for _ in range(n + 1)]

    # 输入中有 n - 1 条边
    # 第 2 个基地到第 n 个基地，每个基地都有一个 a[i]
    index = 1
    for i in range(2, n + 1):
        father = int(data[index])
        index += 1

        # i 和 father 之间有一条道路
        adj[i].append(father)
        adj[father].append(i)

    # ------------------------------------------------------------
    # 第一步：把树以 1 号基地为根，整理出父子关系
    # ------------------------------------------------------------

    parent = [0] * (n + 1)   # parent[x] 表示 x 的父节点
    order = []               # 保存遍历顺序

    stack = [1]
    parent[1] = -1

    while stack:
        u = stack.pop()
        order.append(u)

        for v in adj[u]:
            # 如果 v 是 u 的父节点，就不要再往回走
            if v == parent[u]:
                continue

            parent[v] = u
            stack.append(v)

    # ------------------------------------------------------------
    # 第二步：从叶子节点往根节点处理
    # ------------------------------------------------------------

    INF = 10 ** 9

    need = [-1] * (n + 1)

    down = [INF] * (n + 1)

    answer = 0

    # reversed(order) 就是从叶子往根处理
    for u in reversed(order):

        # forced_build 表示当前节点 u 是否必须建消防局
        forced_build = False

        # min_down 表示 u 的孩子子树中，距离 u 最近的消防局
        min_down = INF

        # child_uncovered 表示 u 是否有孩子没有被覆盖
        child_uncovered = False

        # 查看 u 的所有孩子
        for v in adj[u]:
            if parent[v] != u:
                continue

            # 如果孩子 v 的 need[v] == 1 说明 v 的某个孩子还没被覆盖，那么这个未覆盖点距离当前节点 u 正好是 2
            # 如果不在 u 建消防局，再往 u 的父亲传，距离就变成 3 了，超过消防局的能力范围，所以必须在 u 建消防局
            if need[v] == 1:
                forced_build = True

            # 如果孩子子树里有消防局，那么它到 u 的距离要 +1
            min_down = min(min_down, down[v] + 1)

            # 如果孩子 v 自己还没被覆盖
            if need[v] == 0:
                child_uncovered = True

        # 情况一：必须在当前节点建消防局
        if forced_build:
            answer += 1

            # u 建了消防局，那么 u 的子树中这些危险点都被覆盖了
            need[u] = -1

            # 消防局就在 u，所以距离是 0
            down[u] = 0

        else:
            # 如果最近消防局距离 u 超过 2，那么对 u 来说等于没有消防局能覆盖它
            if min_down > 2:
                min_down = INF

            down[u] = min_down

            # 默认认为 u 的子树都被覆盖
            current_need = -1

            # 如果 u 往下两层内都没有消防局，那么 u 自己没有被覆盖
            if down[u] > 2:
                current_need = 0

            # 如果 u 的某个孩子还没被覆盖
            if child_uncovered:
                # 如果 u 的某个孩子位置上有消防局，也就是 down[u] == 1
                # 那么这个消防局可以通过 u 覆盖其他孩子
                #
                # 例如：
                #     station
                #        |
                #        u
                #       / \
                #   child child
                #
                # 两个孩子之间距离是 2，所以可以覆盖
                if down[u] == 1:
                    pass
                else:
                    # 否则，u 的孩子仍然没被覆盖。对 u 来说，未覆盖点距离自己是 1
                    current_need = max(current_need, 1)

            need[u] = current_need

    # ------------------------------------------------------------
    # 第三步：处理根节点
    # ------------------------------------------------------------
    # 如果最后根节点这边还有未覆盖的点，
    # 那么只能在根节点补建一个消防局
    if need[1] >= 0:
        answer += 1

    print(answer)


if __name__ == "__main__":
    solve()